# Kouman AI — Notebook SFT 1 (`V4.5 Propre Auditée`) : 4 680 Exemples Irréprochables (Zéro Fuite)

Version durcie après audit forensique complet du dataset SFT V1.9 (`V4.3`) — corrige **100 % des 6 défauts mesurés** (4 langues strictes : **Baoulé `bci`, Dioula `dyu`, Français `fr`, Anglais `en`**, **0 % Nouchi / `nusi`**) :

1. **Zéro placeholder corrompu (P0)** : `REGEX_XYZ_OU_BALISE` étendue aux espaces, accents et variantes tronquées (`ZZIKUMANIYAZI`, `ZZKUMANIYAZ`, `XYZPROTÉ14XYZ`, `XYZ ƝANMIƐN SU LƆ 0XYZ`, `Koweri`).
2. **Zéro consigne forcée résiduelle (P0)** : `retirer_suffixes_forces()` coupe jusqu'en fin de chaîne (`Jaabi min be sɔrɔ Dioula kɔnɔ`, `An tɛ su Baoulé nun... Sɛ afɔtuɛ...`, `Jababi la, ntalen...`) avec **assertion bloquante = 0 résidu**.
3. **Identité calibrée LIMA (`100` exemples = `2,1 %`, P0)** : **0 %** de `dyu`/`bci` traduits automatiquement de `identity-v25k` (finis `gbanflɛn kan`, `Koweri`, `ZZIKUMANIYAZI`), uniformisation stricte **`Kouman AI` / `Tree AI Lab`**.
4. **Comportement d'assistant pur en Dioula & Baoulé (P1)** : retrait des chapitres/fils d'Ariane Cosmopedia (`Sapitiri`, `/`, `Aw ni ce an ka`), **12 formulations naturelles** + **variantes clavier/ASR** (`dyu`/`bci` en minuscules sans ponctuation) + dialogues courts d'intentions variées pour éliminer le réflexe d'auto-présentation.
5. **Ponctuation & lexique propres (P1)** : suppression globale des `?` collés en début de phrase Baoulé (`?A ti juejue ?`), blocage des erreurs `Ka an bo su ra` et `Awossi'n ho` pour « Bonne nuit ».
6. **Multi-tour sain & Upload HF fiable (P2)** : déduplication des amorces répétitives dans `trilingues_4tours`, contrôle de langue tour par tour, filtrage des refus IA traduits (`kɛ mɔ n ti AI`), et `HF_HUB_DISABLE_XET=1`.

## 1. Dépendances & Authentification Hugging Face

In [ ]:
%pip install --quiet --upgrade pip
%pip install --quiet --force-reinstall --no-deps "transformers==4.51.3" "huggingface_hub==0.30.2" "tokenizers==0.21.4"
%pip install --quiet --upgrade --upgrade-strategy only-if-needed "pyarrow>=17.0.0" "tqdm" "regex" "safetensors" "pyyaml" "filelock"

In [ ]:
import getpass
import hashlib
import json
import os
import random
import re
from collections import Counter, defaultdict
from pathlib import Path

# Désactive Xet pour éviter les conflits _create_progress_bar lors du push HF
os.environ["HF_HUB_DISABLE_XET"] = "1"

import pyarrow as pa
import pyarrow.parquet as pq
import requests
from huggingface_hub import CommitOperationAdd, HfApi, hf_hub_download
from tqdm.auto import tqdm
from transformers import AutoTokenizer

JETON_HF = os.getenv("HF_TOKEN", "").strip()
if not JETON_HF:
    JETON_HF = getpass.getpass("Jeton Hugging Face (lecture Tree-AI-lab + écriture issoufzousko07) : ").strip()
if not JETON_HF:
    raise PermissionError("[ERREUR] Jeton Hugging Face obligatoire.")
os.environ["HF_TOKEN"] = JETON_HF

client_hf = HfApi(token=JETON_HF)
print(f"Utilisateur Compte HF connecté : {client_hf.whoami()['name']}")

DEPOT_OASST2 = "OpenAssistant/oasst2"
DEPOT_OASST1 = "OpenAssistant/oasst1"
DEPOT_EVERYDAY = "HuggingFaceTB/everyday-conversations-llama3.1-2k"
DEPOT_SMOLTALK = "Tree-AI-lab/smoltalk-multilingue"
DEPOT_SMOLLM = "Tree-AI-lab/smollm-corpus-multilingue"
DEPOT_TRAD_CULTURE = "Tree-AI-lab/kouman-traduction-culture-sft"
DEPOT_TRILINGUES = "Tree-AI-lab/kouman-conversations-trilingues-longues-25k"
DEPOT_IDENTITE = "Tree-AI-lab/kouman-identity-instruct-v25k"

# Nouveau dépôt propre V4.5 (n'écrase PAS V4.3)
DEPOT_SORTIE_SFT = "issoufzousko07/kouman-ia-sft-v4-5-propre"
DEPOT_SFT_PRIVE = True
# Compatible Gemma 2 CPT ou Llama 3.2 1B (changez selon la base à fine-tuner)
MODELE_TOKENIZER = "issoufzousko07/kouman-gemma2-2b-cpt-merged"

GRAINE = 42
TAUX_VALIDATION = 0.05
LONGUEUR_MAX_TOKENS_SFT = 2048  # Étendu à 2048 pour garder intacts les dialogues 4–6 tours

QUOTAS_CATEGORIES = {
    "chat_fr_natif": 780,
    "chat_en_natif": 320,
    "chat_bci_dyu": 1_050,
    "chat_croise_explicite": 100,
    "chat_multitour_relais": 300,
    "traduction_directe": 1_300,
    "melange_bilingue": 730,
    "identite_legere": 100,  # ~2,1 % (calibré LIMA : jamais assez fort pour écraser le mode assistant en Dioula/Baoulé)
}

QUOTAS_TRAD_DIRECTE = {
    ("fr", "dyu"): 220,
    ("dyu", "fr"): 180,
    ("en", "dyu"): 180,
    ("dyu", "en"): 140,
    ("fr", "bci"): 180,
    ("bci", "fr"): 140,
    ("en", "bci"): 130,
    ("bci", "en"): 130,
}

# Prompts STRICTEMENT RÉSERVÉS aux évaluations hors-distribution (V1.8 + nouveaux tests OOD V1.9)
PROMPTS_EVAL_EXCLUS = {
    re.sub(r"\s+", " ", p.strip().lower())
    for p in [
        "Salut l'équipe Kouman, tu peux m'aider aujourd'hui ?",
        "Salut ! Comment ça va aujourd'hui ?",
        "Qui es-tu et qui t'a créé ?",
        "bonjour je suis nicanor et toi ?",
        "bonjour je suis koffi et toi ?",
        "bonjour repond moi en dioula",
        "presente toi en français puis en baoulé",
        "presente toi en francais puis en baoule",
        "combien font 2345+5688",
        "2345 + 5688 ça fait combien ?",
        "Calcule le produit : 37 × 14 = ?",
        "Comment on dit bonjour et bonsoir en baoulé et en dioula ?",
        "Comment on dit merci beaucoup en anglais et en dioula ?",
        "Explique à un débutant comment lire et prononcer le baoulé.",
        "Explique à un débutant comment lire les voyelles spéciales du baoulé.",
        "En baoulé, quel mot dit-on pour accueillir quelqu'un chez soi, et quel mot pour dire bonjour le matin ?",
        "traduis en baoulé : demain matin je vais saluer mes grands-parents au village",
        "traduis en dioula : ma sœur prépare du riz chaud pour le dîner de la famille",
        "traduis en baoulé : la vie est belle au village",
        "traduis en dioula : les parents travaillent pour leurs enfants",
        "et comment on dit je rentre à la maison maintenant en dioula ?",
        "C++ fɔnkisiyɔn `est_pair` sɛbɛn min bɛ `true` segin ni `n` bɛ tila fila la (`n % 2 == 0`).",
        "Je suis débutant, apprends-moi à saluer quelqu'un et dire merci en baoulé avec des exemples concrets.",
        "Fais-moi un mini-cours de dioula pour apprendre à compter de 1 à 5 et saluer le matin.",
        "Quelle est la famille linguistique du baoulé et comment prononcer les lettres ɛ et ɔ ?",
        "Donne-moi 3 phrases simples du quotidien en baoulé avec leur traduction française pour m'entraîner.",
        "Donne-moi 3 phrases simples du quotidien en dioula avec leur traduction française pour m'entraîner.",
        "Traduis en dioula : « La pluie a rendu la terre fertile pour la nouvelle saison. »",
        "Traduis en baoulé : « Mon petit frère apprend à lire vite cette année. »",
        "Traduis en français : « Mɔgɔ bɛɛ ka kan ka baara kɛ ni hɔrɔnya ye walisa dugu ka yiriwa. »",
        "Traduis en français : « Kɛ nzue’n tɔ kpa’n, fie’m be yo klanman, yɛ aliɛ’n sɔn klɔ’n su. »",
        "Translate to Baoulé: The children are reading books at school today.",
        "Translate to Dioula: Good evening, how is your family doing?",
        "Bonsoir ! Je prépare un voyage à Bouaké, donne-moi 3 conseils simples.",
        "Si un agriculteur récolte 45 sacs de cacao le lundi et 38 sacs le mardi, combien de sacs a-t-il en tout ?",
        "Résume en 2 phrases simples l'importance de la coopération dans un village.",
        "comment on dit j'ai faim en baoule",
        "traduit en dioula : j'aime la cote d'ivoire",
        "je veux apprendre le baoule donne moi 3 phrases simples",
        "presente toi brievement",
        "Hakili min be sɔrɔ mɔgɔ bolo, o ye mun lo ye?",
        "Jatikalan kalan n ye.",
        "Nne ka kɛnɛ, n b'a fɛ ka jatebɔ dege",
    ]
}

DOSSIER_TRAVAIL = Path("travail_sft_v4_5_propre")
DOSSIER_TRAVAIL.mkdir(parents=True, exist_ok=True)
DOSSIER_CACHE = DOSSIER_TRAVAIL / "cache_hf"
DOSSIER_CACHE.mkdir(parents=True, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(MODELE_TOKENIZER, token=JETON_HF)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"[OK] Tokenizer prêt : {MODELE_TOKENIZER} | Garde OOD : {len(PROMPTS_EVAL_EXCLUS)} prompts")
print(f"Objectif LIMA V4.5 Propre : {sum(QUOTAS_CATEGORIES.values()):,} exemples irréprochables (0 fuite)")

## 2. Filtres Chirurgicaux V4.5 (Anti-XYZ/ZZ, Anti-Suffixes Forcés, Anti-Nouchi/`nusi`, Anti-OpenAI)

In [ ]:
class ErreurDonneeInvalide(ValueError):
    """Erreur explicite lorsqu'une règle qualité ou anti-fuite échoue."""


# P0 : Bloque TOUT placeholder traduit (XYZPROTÉ14XYZ, XYZ ƝANMIƐN SU LƆ 0XYZ, ZZIKUMANIYAZI, ZZKUMANIYAZ, Koweri, balises)
REGEX_XYZ_OU_BALISE = re.compile(
    r"XYZ|ZZI?KUMAN[A-Za-zÀ-ÿƐƆƝŊ0-9_]*|\b(?:ZZ|XX)[A-Za-zÀ-ÿƐƆƝŊ0-9_]{2,}|__[A-ZÀ-ŸƐƆƝŊ_]{2,}__|"
    r"\bKoweri\b|<like_flɛlɛ>|<tool_call>|</tool_call>|<\|[^|>]*[ɛɔɲŋƐƆƝŊ][^|>]*\|>|<Man ndɛ tɛ>|####\s*\d+",
    re.IGNORECASE,
)
REGEX_PREFIXE_PARASITE = re.compile(
    r"(?:sɛbɛri\s+fɔlɔ|sɛbɛnin\s+fɔlɔ|bayɛlɛmani\s*:|kumadenw\s*:|kumadenw\s+yɛlɛmani|anu\s+oh|anouho)",
    re.IGNORECASE,
)
REGEX_SCRIPT_ETRANGER = re.compile(r"[\u0600-\u06FF\u0400-\u04FF\u4E00-\u9FFF\u3040-\u30FF]")
# Bloque Nouchi dans toutes les langues y compris "nusi" en dioula
REGEX_NOUCHI = re.compile(r"\b(nouchi|nshi|nch|nusi|noussi)\b", re.IGNORECASE)
REGEX_URL = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
REGEX_PERSONA_TIERCE = re.compile(
    r"\b(OpenAI|ChatGPT|Anthropic|Claude|Vicuna|WizardLM|DeepSeek|Google DeepMind|Open Assistant|OpenAssistant|LAION|GPT-4|GPT-3)\b",
    re.IGNORECASE,
)
# P2 : Bloque aussi les refus IA traduits en Baoulé/Dioula et les faux-sens "jeune garçon" pour "startup"
REGEX_PLACEHOLDER_META = re.compile(
    r"(continuez avec l'url|insérez votre|veuillez fournir le texte|je n'ai pas accès à internet|"
    r"as an ai language model|en tant que modèle de langage|en tant qu'ia|as an artificial intelligence|"
    r"kɛ\s+mɔ\s+n\s+ti\s+ai|n\s+ti\s+ai\s+m['’]|gbanflɛn\s+kan\s+kun\s+m['’]ɔ\s+bo\s+junman|"
    r"kanbele(?:\s+dɔ)?\s+ka\s+baara\s+daminɛbaga|"
    r"\[votre |\[insérer|\[nom\]|\[adresse\]|<ce message a été supprimé>)",
    re.IGNORECASE,
)
REGEX_IDENTITE_KOUMAN = re.compile(r"\b(kouman\s*ia|kouman\s*ai|tree\s*ia\s*labs?|tree\s*ai\s*lab)\b", re.IGNORECASE)
REGEX_CAR_BCI_DYU = re.compile(r"[ɛɔɲŋƐƆƝŊ]")
REGEX_MOTS_FR = re.compile(
    r"\b(les|des|une|dans|pour|avec|sont|cette|mais|comme|plus|tout|nous|vous|elle|être|avoir|faire|très|aussi|entre|sans|sous)\b",
    re.IGNORECASE,
)
REGEX_MOTS_EN = re.compile(
    r"\b(the|and|for|with|that|this|from|have|your|about|which|their|would|there|could|should|into|they)\b",
    re.IGNORECASE,
)
# P0 : Coupe jusqu'à la fin de la chaîne (.*$) sans plafond de 100 caractères (nettoie 100 % des 2e phrases Sɛ afɔtuɛ / Jababi la)
REGEX_SUFFIXES_FORCES = [
    re.compile(r"\s*(?:Réponds|Reponds|Merci de répondre)\s+en\s+(?:français|anglais|baoulé|dioula).*$", re.IGNORECASE | re.DOTALL),
    re.compile(r"\s*Tu peux me répondre en (?:français|anglais|baoulé|dioula).*$", re.IGNORECASE | re.DOTALL),
    re.compile(r"\s*(?:Reply|Answer|Respond)\s+in\s+(?:English|French|Baoulé|Dioula).*$", re.IGNORECASE | re.DOTALL),
    re.compile(r"\s*\((?:Make sure your answer is in|Provide the explanation in|All explanations must be in)[^)]*\).*$", re.IGNORECASE | re.DOTALL),
    re.compile(r"\s*(?:Jaabi(?:li)?\s+(?:min\s+be\s+sɔrɔ\s+|di\s+)?Dioula\s+kɔnɔ|Jababi\s+la,?\s+ntalen|An\s+tɛ\s+su\s+(?:Baoulé|Dioula|Wawle)|Amun\s+tɛ\s+su\s+(?:Baoulé|Dioula|Wawle)|Be\s+tɛ\s+su\s+(?:Baoulé|Dioula|Wawle)).*$", re.IGNORECASE | re.DOTALL),
    re.compile(r"\s*Sɛ\s+afɔtuɛ\s+ndɛ['’]n.*$", re.IGNORECASE | re.DOTALL),
]
# Assertion finale anti-résidu : aucun de ces marqueurs synthétiques ne doit survivre dans un prompt user
REGEX_RESIDU_SUFFIXE_INTERDIT = re.compile(
    r"(?:Jaabi(?:li)?\s+(?:min\s+be\s+sɔrɔ\s+|di\s+)?Dioula\s+kɔnɔ|Jababi\s+la,?\s+ntalen|"
    r"(?:An|Amun|Be)\s+tɛ\s+su\s+(?:Baoulé|Dioula|Wawle)|Sɛ\s+afɔtuɛ\s+ndɛ['’]n)",
    re.IGNORECASE,
)
REGEX_POINT_INTERRO_COLLE = re.compile(r"(^|[\s.!?,:;])\?(?=[A-Za-zÀ-ÿɛɔɲŋƐƆƝŊ])")

rejets: Counter = Counter()
registre_exemples: dict[str, list[dict]] = {cat: [] for cat in QUOTAS_CATEGORIES}
hashes_vus: set[str] = set()


def nettoyer_texte(texte: str) -> str:
    if not isinstance(texte, str):
        return ""
    t = texte.replace("\xa0", " ").replace("\r\n", "\n").replace("<br/>", "\n").replace("<br>", "\n")
    # P1 : Supprime '?' collé devant un mot PARTOUT (début de ligne ET milieu de ligne après ! ou .)
    t = REGEX_POINT_INTERRO_COLLE.sub(r"\1", t)
    t = re.sub(r"[ \t]{2,}", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()


def retirer_suffixes_forces(texte: str) -> str:
    resultat = texte
    for _ in range(2):
        for reg in REGEX_SUFFIXES_FORCES:
            candidat = reg.sub("", resultat).strip()
            if len(candidat) >= 8:
                resultat = candidat
    return resultat


RNG_CLAVIER = random.Random(GRAINE + 2026)


def style_clavier_naturel(prompt_user: str, langue: str, prob: float = 0.50) -> str:
    """Transforme une fraction des prompts utilisateur (FR 50 %, DYU/BCI 35 %) en style clavier/ASR réel
    (minuscules, sans ponctuation finale, apostrophe ASCII). Ne modifie JAMAIS les réponses assistant.
    """
    if "\n" in prompt_user or len(prompt_user) > 220:
        return prompt_user

    if langue in ("dyu", "bci"):
        if RNG_CLAVIER.random() > 0.35:
            return prompt_user
        t = prompt_user.strip()
        if RNG_CLAVIER.random() < 0.70 and t:
            t = t[0].lower() + t[1:]
        if RNG_CLAVIER.random() < 0.65:
            t = t.rstrip(".?!")
        if RNG_CLAVIER.random() < 0.60:
            t = t.replace("’", "'")
        t = re.sub(r"\s{2,}", " ", t).strip()
        if re.sub(r"\s+", " ", t.lower()) in PROMPTS_EVAL_EXCLUS:
            return prompt_user
        return t if len(t) >= 4 else prompt_user

    if langue != "fr" or RNG_CLAVIER.random() > prob:
        return prompt_user
    t = prompt_user.strip()
    t = re.sub(r'[«»"]\s*', "", t)
    t = re.sub(r"\s+([?!:;])", r"\1", t)
    t = re.sub(r"\bComment dit-on\b", RNG_CLAVIER.choice(["comment on dit", "comment dit on", "Comment on dit"]), t, flags=re.IGNORECASE)
    t = re.sub(r"\bTraduis\b", RNG_CLAVIER.choice(["traduis", "traduit", "Traduit"]), t)
    if RNG_CLAVIER.random() < 0.45:
        t = t.replace("baoulé", "baoule").replace("Baoulé", "baoule")
    if RNG_CLAVIER.random() < 0.30:
        t = t.replace("français", "francais").replace("Français", "francais")
    if RNG_CLAVIER.random() < 0.35:
        t = t.replace("Présente-toi", "presente toi").replace("Qui es-tu", "qui es tu")
    if RNG_CLAVIER.random() < 0.65 and t and t[0].isupper() and not t.startswith(("C++", "Python")):
        t = t[0].lower() + t[1:]
    if RNG_CLAVIER.random() < 0.45:
        t = t.rstrip(".?")
    t = re.sub(r"\s{2,}", " ", t).strip()
    if re.sub(r"\s+", " ", t.lower()) in PROMPTS_EVAL_EXCLUS:
        return prompt_user
    return t if len(t) >= 4 else prompt_user


def ratio_correspondances(reg: re.Pattern, texte: str) -> float:
    mots = re.findall(r"[A-Za-zÀ-ÿɛɔɲŋƐƆƝŊ']+", texte)
    return (len(reg.findall(texte)) / len(mots)) if mots else 0.0


def contient_code_bci_corrompu(texte: str) -> bool:
    for bloc in re.findall(r"```[a-zA-Z0-9_+-]*\n(.*?)```", texte, re.DOTALL):
        lignes_code = [l for l in bloc.splitlines() if not l.strip().startswith(("#", "//", "/*", "*"))]
        if REGEX_CAR_BCI_DYU.search("\n".join(lignes_code)):
            return True
    return False


def detecter_motif_rejet(
    texte: str,
    langue_attendue: str | None = None,
    autoriser_identite: bool = False,
    autoriser_mention_tierce_refus: bool = False,
) -> str | None:
    if not texte or len(texte.strip()) < 2:
        return "texte_vide"
    if REGEX_XYZ_OU_BALISE.search(texte):
        return "xyz_balise_ou_gsm8k"
    if REGEX_PREFIXE_PARASITE.search(texte):
        return "prefixe_parasite_trad"
    if REGEX_SCRIPT_ETRANGER.search(texte):
        return "script_etranger"
    if REGEX_NOUCHI.search(texte):
        return "nouchi_interdit"
    if REGEX_URL.search(texte):
        return "url_brute"
    if not autoriser_mention_tierce_refus and REGEX_PERSONA_TIERCE.search(texte):
        return "persona_ia_tierce"
    if REGEX_PLACEHOLDER_META.search(texte):
        return "placeholder_meta"
    if not autoriser_identite and REGEX_IDENTITE_KOUMAN.search(texte):
        return "identite_hors_categorie"
    if "```" in texte and contient_code_bci_corrompu(texte):
        return "code_bci_corrompu"
    if langue_attendue and len(texte) >= 65 and "```" not in texte:
        if langue_attendue in ("bci", "dyu"):
            nb_spec = len(REGEX_CAR_BCI_DYU.findall(texte))
            if ratio_correspondances(REGEX_MOTS_FR, texte) >= 0.14 and nb_spec < max(2, len(texte) // 120):
                return f"fuite_fr_dans_{langue_attendue}"
            if ratio_correspondances(REGEX_MOTS_EN, texte) >= 0.16:
                return f"fuite_en_dans_{langue_attendue}"
        elif langue_attendue == "fr":
            if ratio_correspondances(REGEX_MOTS_EN, texte) >= 0.18:
                return "fuite_en_dans_fr"
    return None


def normaliser_messages_chat(messages_bruts: list[dict]) -> list[dict] | None:
    if not messages_bruts:
        return None
    sys_morceaux: list[str] = []
    tours: list[dict] = []
    for m in messages_bruts:
        if not isinstance(m, dict):
            return None
        role_brut = str(m.get("role") or m.get("from") or "").strip().lower()
        contenu = nettoyer_texte(str(m.get("content") or m.get("value") or ""))
        if not contenu:
            return None
        if role_brut == "system":
            sys_morceaux.append(contenu)
            continue
        if role_brut in ("user", "human", "prompter"):
            role = "user"
        elif role_brut in ("assistant", "model", "gpt"):
            role = "assistant"
        else:
            return None
        if tours and tours[-1]["role"] == role:
            return None
        tours.append({"role": role, "content": contenu})
    if len(tours) < 2 or tours[0]["role"] != "user":
        return None
    if tours[-1]["role"] != "assistant":
        tours = tours[:-1]
    if len(tours) < 2 or len(tours) % 2 != 0:
        return None
    if sys_morceaux:
        tours[0]["content"] = "\n".join(sys_morceaux) + "\n\n" + tours[0]["content"]
    return tours


def longueur_tokens_chat(messages: list[dict]) -> int:
    texte = "".join(
        f"<start_of_turn>{'user' if m['role'] == 'user' else 'model'}\n{m['content']}<end_of_turn>\n"
        for m in messages
    )
    return len(tokenizer.encode(texte, add_special_tokens=True))


def ajouter_dialogue_sft(
    categorie: str,
    messages_bruts: list[dict],
    langue_consigne: str,
    langue_reponse: str,
    source: str,
    verifier_langue_reponse: bool = True,
    autoriser_identite: bool = False,
    autoriser_mention_tierce_refus: bool = False,
    appliquer_clavier: bool = True,
) -> bool:
    if categorie not in registre_exemples:
        raise ErreurDonneeInvalide(f"Catégorie inconnue : {categorie}")
    messages = normaliser_messages_chat(messages_bruts)
    if not messages:
        rejets["format_invalide"] += 1
        return False

    for m in messages:
        if m["role"] == "user":
            if re.sub(r"\s+", " ", m["content"].strip().lower()) in PROMPTS_EVAL_EXCLUS:
                rejets["exclu_ood"] += 1
                return False
            if appliquer_clavier and langue_consigne in ("fr", "dyu", "bci") and source not in ("salutation_fr_clavier", "bascule_langue_curatee", "identite_curatee", "assistant_dyu_bci_curate"):
                m["content"] = style_clavier_naturel(m["content"], langue_consigne, prob=0.50)
            if REGEX_RESIDU_SUFFIXE_INTERDIT.search(m["content"]):
                rejets["user_suffixe_force_residu"] += 1
                return False
            if re.sub(r"\s+", " ", m["content"].strip().lower()) in PROMPTS_EVAL_EXCLUS:
                rejets["exclu_ood"] += 1
                return False
            motif_u = detecter_motif_rejet(
                m["content"],
                None,
                autoriser_identite=autoriser_identite,
                autoriser_mention_tierce_refus=autoriser_mention_tierce_refus,
            )
            if motif_u:
                rejets[f"user_{motif_u}"] += 1
                return False
        else:
            if len(m["content"]) < 3:
                rejets["reponse_vide"] += 1
                return False
            lg_ctrl = langue_reponse if (verifier_langue_reponse and langue_reponse in ("bci", "dyu", "fr")) else None
            motif_a = detecter_motif_rejet(
                m["content"],
                lg_ctrl,
                autoriser_identite=autoriser_identite,
                autoriser_mention_tierce_refus=autoriser_mention_tierce_refus,
            )
            if motif_a:
                rejets[f"asst_{motif_a}"] += 1
                return False

    while len(messages) > 2 and longueur_tokens_chat(messages) > LONGUEUR_MAX_TOKENS_SFT:
        messages = messages[:-2]
    nb_tok = longueur_tokens_chat(messages)
    if nb_tok > LONGUEUR_MAX_TOKENS_SFT:
        rejets["trop_long"] += 1
        return False

    morceaux = [m["role"] + ":" + re.sub(r"\s+", " ", m["content"].lower()) for m in messages]
    empreinte = hashlib.sha256("||".join(morceaux).encode("utf-8")).hexdigest()
    if empreinte in hashes_vus:
        rejets["doublon"] += 1
        return False
    hashes_vus.add(empreinte)

    registre_exemples[categorie].append(
        {
            "id": empreinte[:16],
            "categorie": categorie,
            "langue_consigne": langue_consigne,
            "langue_reponse": langue_reponse,
            "source": source,
            "messages": messages,
            "nb_tokens": nb_tok,
        }
    )
    return True


print("[OK] Filtres chirurgicaux V4.3 (Kouman AI 1.9) prêts.")

## 3. Blocs 1 & 2 : Chat FR Natif (jusqu'à 6 tours + Bascules de Langue) & Chat EN Multi-Tour

In [ ]:
# 3A. Reconstruction des meilleures branches françaises humaines (jusqu'à 6 tours = 12 messages)
def extraire_dialogues_oasst_fr(repo_id: str, nom_parquet: str, tag_source: str) -> list[list[dict]]:
    print(f"Téléchargement FR natif : {repo_id}")
    chemin_p = hf_hub_download(
        repo_id=repo_id,
        filename=nom_parquet,
        repo_type="dataset",
        token=JETON_HF,
        local_dir=str(DOSSIER_CACHE / tag_source),
    )
    cols = ["message_id", "parent_id", "text", "role", "lang", "review_result", "deleted", "rank"]
    lignes = pq.read_table(chemin_p, columns=cols).to_pylist()
    enfants: dict[str, list[dict]] = defaultdict(list)
    racines: list[dict] = []

    for row in lignes:
        if row.get("lang") != "fr" or row.get("deleted") is True or row.get("review_result") is False:
            continue
        pid = row.get("parent_id")
        if pid is None:
            if row.get("role") == "prompter":
                racines.append(row)
        else:
            enfants[str(pid)].append(row)

    dialogues: list[list[dict]] = []
    for rac in racines:
        # Débloqué jusqu'à 6 échanges (12 messages) pour renforcer la tenue multi-tour
        chemin_msgs: list[dict] = [{"role": "user", "content": str(rac["text"])}]
        courant = rac
        for _ in range(11):
            fils = enfants.get(str(courant["message_id"]), [])
            if not fils:
                break
            role_attendu = "assistant" if courant["role"] == "prompter" else "prompter"
            candidats = [f for f in fils if f.get("role") == role_attendu]
            if not candidats:
                break
            candidats.sort(key=lambda x: (999 if x.get("rank") is None else float(x["rank"])))
            meilleur = candidats[0]
            if role_attendu == "assistant" and meilleur.get("rank") is not None and float(meilleur["rank"]) > 0.0:
                break
            chemin_msgs.append({"role": "assistant" if role_attendu == "assistant" else "user", "content": str(meilleur["text"])})
            courant = meilleur
        if len(chemin_msgs) >= 2:
            if chemin_msgs[-1]["role"] != "assistant":
                chemin_msgs = chemin_msgs[:-1]
            if len(chemin_msgs) >= 2:
                dialogues.append(chemin_msgs)
    return dialogues


rng_fr = random.Random(GRAINE + 10)

# Salutations courtes naturelles FR au clavier
SALUTATIONS_FR_CLAVIER = [
    ("bonjour", "Bonjour ! Comment puis-je vous aider aujourd'hui ?"),
    ("Bonjour !", "Bonjour ! Que puis-je faire pour vous ?"),
    ("salut", "Salut ! Comment puis-je t'aider ?"),
    ("bonsoir", "Bonsoir ! En quoi puis-je vous être utile ?"),
    ("coucou", "Coucou ! Dis-moi comment je peux t'aider."),
    ("ça va ?", "Très bien, merci ! Et toi, comment vas-tu ?"),
    ("ca va", "Ça va très bien, merci ! Que puis-je faire pour toi ?"),
    ("bonjour tu peux m'aider", "Bonjour ! Bien sûr, je t'écoute. De quoi as-tu besoin ?"),
    ("salut tu vas bien", "Salut ! Oui, je vais bien, merci. Comment puis-je t'aider ?"),
    ("bonsoir j'ai une question", "Bonsoir ! Je t'écoute, pose-moi ta question."),
    ("hello ça va ?", "Bonjour ! Oui ça va très bien, merci. Comment puis-je t'aider ?"),
    ("bonjour comment ca va", "Bonjour ! Ça va bien, merci. En quoi puis-je vous aider ?"),
    ("merci beaucoup", "Avec plaisir ! N'hésite pas si tu as d'autres questions."),
    ("merci", "De rien ! Je reste à ta disposition."),
    ("ok merci", "Avec plaisir ! Bonne continuation."),
    ("je te pose une question", "Vas-y, je t'écoute ! Quelle est ta question ?"),
]
for u_sal, a_sal in SALUTATIONS_FR_CLAVIER:
    ajouter_dialogue_sft(
        "chat_fr_natif",
        [{"role": "user", "content": u_sal}, {"role": "assistant", "content": a_sal}],
        "fr", "fr", "salutation_fr_clavier", verifier_langue_reponse=True,
    )

# Consignes explicites de bascule de langue & double-langue (100 % conversationnel propre, zéro article web)
BASCULES_ET_DOUBLES_CURATEES = [
    # --- FR -> DYU (salutations, accueil & thèmes quotidiens) ---
    ("salut reponds moi en dioula", "I ni sɔgɔma ! N bɛ se ka i dɛmɛ cogo di bi ? I ka ɲiningali fɔ n’ ye, n bɛna i jaabi dioulakan na.", "fr", "dyu", False),
    ("Bonjour, peux-tu me répondre en dioula s'il te plaît ?", "I ni tile ! Ɔwɔ, n bɛ se ka kuma n’i ye dioulakan na. I b’a fɛ an ka baro kɛ mun kan bi ?", "fr", "dyu", False),
    ("bonjour parle moi en dioula", "I ni sɔgɔma ! Ayiwa, an ka baro kɛ dioulakan na. I b’a fɛ ka mun lo lɔn bi ?", "fr", "dyu", False),
    ("salut tu peux me repondre en dioula ?", "I ni tile ! Ɔwɔ, n bɛ se ka i jaabi dioulakan na. I ka ɲiningali kɛ !", "fr", "dyu", False),
    ("coucou repond en dioula", "I ni tile ! I ka kɛnɛ wa ? N bɛ yan ka i dɛmɛ dioulakan na.", "fr", "dyu", False),
    ("parle moi en dioula maintenant", "Ayiwa, an bɛna kuma dioulakan na sisan ! I b’a fɛ ka mun lo lɔn wala ka mun dege bi ?", "fr", "dyu", False),
    ("bonsoir reponds en dioula", "I ni wula ! I ka kɛnɛ wa ? Fɔ n’ ye n bɛ se ka i dɛmɛ cogo min na bi wula fɛ.", "fr", "dyu", False),
    ("bonjour comment tu vas ? reponds en dioula", "I ni sɔgɔma ! N ka kɛnɛ kosɔbɛ, barika. E do, i ka kɛnɛ wa ?", "fr", "dyu", False),
    ("reponds moi en dioula : que fais tu aujourd'hui ?", "Bi, n bɛ yan walisa ka mɔgɔw dɛmɛ u ka ɲiningaliw, u ka kalanw ani u ka kan bayɛlɛmaliw la.", "fr", "dyu", False),
    ("parle moi de la cote d'ivoire en dioula", "Kɔdiwari ye Afiriki tilebinfan jamana dɔ ye min tɔgɔ bɔra a ka sɛnɛkɛ, a ka kakawɔ, a ka laadalakow ani a ka dugubaw kosɔn i n’a fɔ Abijan ni Yamusukro ani Bwake. Mɔgɔ suguya caaman bɛ sigi yen hɛɛrɛ ni bɛn kɔnɔ.", "fr", "dyu", False),
    ("parle moi de l'agriculture en dioula", "Sɛnɛkɛ ye baara jɔnjɔn ye Afiriki tilebinfan na. A bɛ dumuni di denbayaw ma i n’a fɔ malo, kaba, ku ani banaan, ka sɔrɔ kakawɔ ni kafe ni kɔɔri bɛ wari lase jamana kɔnɔ.", "fr", "dyu", False),
    ("donne moi des conseils pour bien etudier en dioula", "Walisa ka kalan kɛ ka ɲɛ, i ka kan ka : 1) wagati latigɛ loon o loon ka i ka kalansenw segi, 2) yɔrɔ makunnen sugandi min tɛ mɔgɔ hakili ɲagami, 3) ɲiningaliw kɛ ni i ma koo dɔ faamu, ani 4) sunɔgɔ ka ɲɛ walisa i hakili ka lafiya.", "fr", "dyu", False),
    ("explique l'importance de la sante en dioula", "Kɛnɛya ka fisa ni wari ye. Walisa ka kɛnɛya mara, an ka kan ka dumuni ɲuman dumu, ka ji sɛniman min, ka farikoloɲɛnajɛ kɛ ani ka lafiya ka ɲɛ su fɛ.", "fr", "dyu", False),
    ("parle moi du travail en equipe en dioula", "Baara kɛli ɲɔgɔn fɛ bɛ fanga di mɔgɔw ma. Ni bɛɛ b’a ka dɔnniya n’a ka seko fara ɲɔgɔn kan, baara bɛ kɛ joona ani a bɛ ɲɛ ka tɛmɛ kelen-kelen baara kan.", "fr", "dyu", False),
    ("parle moi de la famille en dioula", "Denbaya ye adamaden ka sigida jɔnjɔn ye. Denbaya kɔnɔ, denmisɛnw bɛ bonya, kanuya, dɛmɛɲɔgɔnya ani baara kɛcogo dege u bangebagaw fɛ.", "fr", "dyu", False),
    ("donne moi un conseil pour bien cuisiner en dioula", "Walisa ka tobili kɛ ka ɲɛ, i ka kan ka i ka fɛnw bɛɛ ko ka sɛniya fɔlɔ, ka ta hakɛ bɛnnen bila a la, ani ka a nɛnɛ lajɛ sanni i k’a bɔ ta kan.", "fr", "dyu", False),
    # --- FR -> BCI (salutations, accueil & thèmes quotidiens) ---
    ("salut reponds moi en baoule", "Aɲiho ! Ɛɛn, n kwla tɛ ɔ su wawle nun. ?Ngue sa yɛ a kunndɛ kɛ n uka wɔ su andɛ ɔ ?", "fr", "bci", False),
    ("bonjour repond moi en baoule", "Aɲiho ! Akwaba. N kwla koko yalɛ nin wɔ wawle nun. ?Ngue ndɛ yɛ a le i andɛ ɔ ?", "fr", "bci", False),
    ("Bonjour, peux-tu me répondre en baoulé ?", "Aɲiho ! Akwaba. N kwla koko yalɛ nin wɔ wawle nun. ?Ngue ndɛ yɛ a kunndɛ kɛ e koko su yalɛ andɛ ɔ ?", "fr", "bci", False),
    ("parle moi en baoule s'il te plait", "Ɛɛn, kɛ é kókó yalɛ wawle nun dɔ nga su ! Kan like nga a kunndɛ kɛ a si i’n kle min.", "fr", "bci", False),
    ("coucou tu peux parler en baoule ?", "Aɲiho ! Ɛɛn, n kwla ijɔ wawle nun. Usa min kosan nga a klo’n.", "fr", "bci", False),
    ("bonsoir reponds en baoule", "Awossi'n o ! ?A ti juejue ? Kan like nga n kwla yo man wɔ nnɔsua nga nun’n kle min.", "fr", "bci", False),
    ("bonjour comment ca va ? reponds en baoule", "Aɲiho ! N ti juejue kpa, mo. ?Yɛ wɔ li, a ti juejue ?", "fr", "bci", False),
    ("parle moi de la cote d'ivoire en baoule", "Kɔti Divwari ti nvle kun m’ɔ o Afriki lɔ’n. Be si i kpa i fie dilɛ’n, i kakao’n, i kafe’n, nin i klɔ dan kɛ Abijan, Yamusukro, nin Bwake sa’m be ti. Sran wafawafa kpanngban be tran nun alaje nin anuanzɛ nun.", "fr", "bci", False),
    ("donne moi des conseils pour reussir a l'ecole en baoule", "Sɛ a kunndɛ kɛ a yo kpa suklu lɔ’n, ɔ fata kɛ : 1) a kanngan ɔ fluwa’m be nun cɛn kwlaa, 2) a sie ɔ su kpa kɛ like klefuɛ’n su ijɔ’n, 3) a usa kosan sɛ a wunman ndɛ kun i wlɛ’n, yɛ 4) a lafi kpa kɔnguɛ naan ɔ ti nun’n w’a yo weiin.", "fr", "bci", False),
    ("parle moi de la famille en baoule", "Awlobo’n ti like cinnjin kpa e mɛn dilɛ’n nun. Awlobo nun yɛ ba’m be suan aɲinzɛ, sran bulɛ, nin junman dilɛ ɔ. Kɛ awlobofuɛ’m be uka be wun’n, be kwla jran sa kekle’m be ɲrun kekle.", "fr", "bci", False),
    ("parle moi de l'agriculture en baoule", "Fie dilɛ’n ti junman dan kpa Kɔti Divwari lɔ. Fie nun yɛ be ɲan aliɛ kɛ duɔ, avie, banda nin maniɔku sa ɔ, yɛ kakao nin kafe be man sika klɔ’n su.", "fr", "bci", False),
    ("explique l'importance de la sante en baoule", "Juejue tranlɛ’n ti kpa tra sika. Sɛ a kunndɛ kɛ a tran juejue’n, di aliɛ kpa, nɔn nzue ng’ɔ ti sanwun’n, yɛ lafi kpa kɔnguɛ.", "fr", "bci", False),
    ("parle moi du respect des anciens en baoule", "E nannan’m be bulɛ’n ti aɲinzɛ kpa e klɔ’m be su. Sran kpɛnngbɛn’m be si mɛn’n nun sa kpanngban, ɔ maan be afɔtuɛ’m be uka gbanflɛn nin talua mun.", "fr", "bci", False),
    ("donne moi un conseil pour bien cuisiner en baoule", "Kɛ á tɔ́n aliɛ’n, wunnzin ninnge kwlaa ka naan w’a bo i bo, nian sin’n i kpa, yɛ nian sɛ njin’n w’a ju ka naan w’a yi i sin’n su.", "fr", "bci", False),
    # --- DOUBLES LANGUES PROPRES (FR puis BCI / FR puis DYU / EN puis BCI / EN puis DYU) ---
    ("salue moi en français puis en baoulé", "**Français :** Bonjour ! J'espère que tu passes une excellente journée. Comment puis-je t'aider ?\n\n**Baoulé :** Aɲiho ! N srɛ kɛ ɔ cɛn’n yo klanman. ?Ngue sa yɛ n kwla uka wɔ su andɛ ɔ ?", "fr", "multi", False),
    ("salue moi en français puis en dioula", "**Français :** Bonjour ! J'espère que tu vas très bien aujourd'hui. En quoi puis-je t'aider ?\n\n**Dioula :** I ni sɔgɔma ! N bɛ jigi ko i ka kɛnɛ kosɔbɛ bi. N bɛ se ka i dɛmɛ cogo di ?", "fr", "multi", False),
    ("souhaite la bienvenue en français puis en baoulé", "**Français :** Bienvenue chez nous ! Nous sommes très heureux de vous accueillir.\n\n**Baoulé :** Akwaba ! E klun jɔ kpa kɛ e su sɔ amun nun andɛ.", "fr", "multi", False),
    ("souhaite la bienvenue en français puis en dioula", "**Français :** Bienvenue ! Nous sommes ravis de vous recevoir aujourd'hui.\n\n**Dioula :** Aw ni sɛnɛ ! Bisimila ! An nisɔn ka di kosɔbɛ ka aw kunbɛn bi.", "fr", "multi", False),
    ("dis merci et bonne journée en français puis en baoulé", "**Français :** Merci beaucoup pour votre aide et passez une très belle journée !\n\n**Baoulé :** A yo ɔ mo kpa ɔ ukalɛ’n ti, yɛ ɔ cɛn’n yo klanman !", "fr", "multi", False),
    ("dis merci et bonne journée en français puis en dioula", "**Français :** Merci beaucoup pour ton aide et passe une excellente journée !\n\n**Dioula :** I ni ce kosɔbɛ i ka dɛmɛ kosɔn, ani i ka tile ka diya !", "fr", "multi", False),
    ("donne un conseil pour réussir à l'école en français puis en baoulé", "**Français :** Pour réussir à l'école, révise tes leçons chaque jour et n'hésite pas à poser des questions quand tu ne comprends pas.\n\n**Baoulé :** Sɛ a kunndɛ kɛ a yo kpa suklu lɔ’n, kanngan ɔ fluwa’m be nun cɛn kwlaa yɛ usa kosan kɛ a wunman like kun i wlɛ’n.", "fr", "multi", False),
    ("donne un conseil pour réussir à l'école en français puis en dioula", "**Français :** Pour réussir à l'école, révise régulièrement tes cours et pose des questions dès qu'un point n'est pas clair.\n\n**Dioula :** Walisa ka ɲɛtaa sɔrɔ kalanso la, i ka kalansenw segi loon o loon ani ɲiningaliw kɛ ni i ma koo dɔ faamu.", "fr", "multi", False),
    ("parle de l'importance de la famille en français puis en baoulé", "**Français :** La famille est un pilier essentiel : elle nous transmet les valeurs, le soutien et la solidarité au quotidien.\n\n**Baoulé :** Awlobo’n ti like cinnjin kpa : awlobo nun yɛ e suan aɲinzɛ, sran bulɛ nin be wun ukalɛ cɛn kwlaa ɔ.", "fr", "multi", False),
    ("parle de l'importance de la famille en français puis en dioula", "**Français :** La famille est notre premier soutien : elle nous apprend le respect, l'entraide et le courage dans la vie.\n\n**Dioula :** Denbaya ye an ka kɔkɔrɔdonbaga fɔlɔ ye : a bɛ bonya, dɛmɛɲɔgɔnya ani jija dege an na ɲɛnamaya kɔnɔ.", "fr", "multi", False),
    ("écris un court message d'encouragement en français et en baoulé", "**Français :** Courage ! Avec de la patience et du travail régulier, tu atteindras tes objectifs.\n\n**Baoulé :** Tra ɔ awlɛn ! Sɛ a tra ɔ awlɛn naan a di junman kpa’n, á ɲán like nga a kunndɛ’n.", "fr", "multi", False),
    ("écris un court message d'encouragement en français et en dioula", "**Français :** Ne lâche rien ! La persévérance et le travail finissent toujours par payer.\n\n**Dioula :** I jija ! Mɔgɔ min bɛ muɲuli kɛ ka baara kɛ ni cɛsiri ye, o bɛ ɲɛtaa sɔrɔ tuma bɛɛ.", "fr", "multi", False),
    ("presente toi en francais et en dioula", "**Français :** Bonjour ! Je suis **Kouman AI**, un assistant développé par **Tree AI Lab** en Côte d'Ivoire pour aider en français, anglais, baoulé et dioula.\n\n**Dioula :** I ni sɔgɔma ! Ne ye **Kouman AI** ye, hakilidiya dɛmɛbaga min dilanna **Tree AI Lab** fɛ Kɔdiwari walisa ka mɔgɔw dɛmɛ faransikan, angɛlɛkan, bawulɛkan ani dioulakan na.", "fr", "multi", True),
    ("presente toi en anglais puis en baoule", "**English:** Hello! I am **Kouman AI**, a multilingual AI assistant developed by **Tree AI Lab** in Côte d'Ivoire (Baoulé, Dioula, French, and English).\n\n**Baoulé :** Aɲiho ! Be flɛ min kɛ **Kouman AI**, **Tree AI Lab** m’ɔ o Kɔti Divwari lɔ’n yɛ ɔ yili min naan n uka sran mun wawle, diula, flanse nin angle nun ɔ.", "fr", "multi", True),
    ("introduce yourself in English and then in Dioula", "**English:** Hello! I am **Kouman AI**, created by **Tree AI Lab** in Côte d'Ivoire to support Baoulé, Dioula, French, and English.\n\n**Dioula:** I ni sɔgɔma! Ne ye **Kouman AI** ye, **Tree AI Lab** lo ye n dilan Kɔdiwari jamana na.", "en", "multi", True),
    ("introduce yourself in English and then in Baoulé", "**English:** Hello! I am **Kouman AI**, created by **Tree AI Lab** in Côte d'Ivoire to support Baoulé, Dioula, French, and English.\n\n**Baoulé:** Aɲiho! Be flɛ min kɛ **Kouman AI**, **Tree AI Lab** (Kɔti Divwari) yɛ ɔ yili min ɔ.", "en", "multi", True),
    # --- EN -> DYU / EN -> BCI / DYU & BCI -> FR & EN ---
    ("Hello, please reply in Dioula.", "I ni sɔgɔma ! N ka kɛnɛ, n bɛ yan ka i dɛmɛ dioulakan na. I b’a fɛ ka mun lɔn bi ?", "en", "dyu", False),
    ("Hi, can you answer me in Baoulé?", "Aɲiho ! Ɛɛn, n kwla tɛ ɔ su wawle nun. ?Ngue sa yɛ a kunndɛ kɛ n uka wɔ su andɛ ɔ ?", "en", "bci", False),
    ("Talk about education in Dioula.", "Kalan ye fɛɛn jɔnjɔn ye jamana yiriwali kama. A bɛ dɔnniya di denmisɛnw ma ani k’u labɛn siniɲɛsigi ɲuman kama.", "en", "dyu", False),
    ("Talk about education in Baoulé.", "Suklu dilɛ’n ti like cinnjin kpa nvle kun i ɲrun kɔlɛ’n ti. Ɔ man ba’m be ngwlɛlɛ yɛ ɔ siesie be ainman mɛn dilɛ’n ti.", "en", "bci", False),
    ("Give me 2 health tips in Dioula.", "Kɛnɛya ladili fila : 1) Ji sɛniman caaman min loon o loon. 2) Dumuni suguya caman dumu i n’a fɔ nakɔfɛnw ni yiridenw.", "en", "dyu", False),
    ("Give me 2 health tips in Baoulé.", "Juejue tranlɛ afɔtuɛ nɲɔn : 1) Nɔn nzue sanwun kpanngban cɛn kwlaa. 2) Di mma nin nɲanman kpa naan ɔ wunnɛn’n yo kekle.", "en", "bci", False),
    ("I ni sɔgɔma, réponds-moi en français s'il te plaît.", "Bonjour ! Bien sûr, je te réponds en français. Comment puis-je t'aider aujourd'hui ?", "dyu", "fr", False),
    ("Aɲiho, réponds-moi en français s'il te plaît.", "Bonjour ! Avec plaisir, je te réponds en français. De quoi souhaites-tu parler aujourd'hui ?", "bci", "fr", False),
    ("I ni tile, kuma n’ ye kɛnɛya kan faransikan na.", "Bonjour ! La santé est notre bien le plus précieux : bien manger, boire de l'eau potable, dormir suffisamment et bouger régulièrement permettent de rester en forme.", "dyu", "fr", False),
    ("Aɲiho, kan suklu dilɛ’n i ndɛ kle min flanse nun.", "Bonjour ! L'école est une chance précieuse : elle permet d'apprendre à lire, compter, réfléchir et préparer son avenir professionnel.", "bci", "fr", False),
    ("I ni sɔgɔma, please answer me in English.", "Good morning! Of course, I am happy to help you in English. What would you like to talk about today?", "dyu", "en", False),
    ("Aɲiho, please answer me in English.", "Good morning! Sure, I can help you in English. How can I assist you today?", "bci", "en", False),
]
for u_basc, a_basc, lg_u, lg_a, est_id in BASCULES_ET_DOUBLES_CURATEES:
    ajouter_dialogue_sft(
        "chat_croise_explicite",
        [{"role": "user", "content": u_basc}, {"role": "assistant", "content": a_basc}],
        lg_u, lg_a, "bascule_et_double_curatee",
        verifier_langue_reponse=(lg_a in ("fr", "bci", "dyu")),
        autoriser_identite=est_id,
        appliquer_clavier=False,
    )

dialogues_fr = (
    extraire_dialogues_oasst_fr(DEPOT_OASST2, "data/train-00000-of-00001-88ba0162028a73fc.parquet", "oasst2")
    + extraire_dialogues_oasst_fr(DEPOT_OASST2, "data/validation-00000-of-00001-1deeef95c3248fe0.parquet", "oasst2_val")
    + extraire_dialogues_oasst_fr(DEPOT_OASST1, "data/train-00000-of-00001-b42a775f407cee45.parquet", "oasst1")
)
# Priorité aux vrais dialogues multi-tours FR (2 à 6 tours) avant de compléter en mono-tour
dlgs_fr_multi = [d for d in dialogues_fr if len(d) >= 4]
dlgs_fr_mono = [d for d in dialogues_fr if len(d) == 2]
rng_fr.shuffle(dlgs_fr_multi)
rng_fr.shuffle(dlgs_fr_mono)

nb_fr_multi = 0
for dlg in dlgs_fr_multi + dlgs_fr_mono:
    if len(registre_exemples["chat_fr_natif"]) >= QUOTAS_CATEGORIES["chat_fr_natif"]:
        break
    if ajouter_dialogue_sft("chat_fr_natif", dlg, "fr", "fr", "oasst_fr_natif", verifier_langue_reponse=True):
        if len(dlg) >= 4:
            nb_fr_multi += 1

print(f"[OK] Chat FR natif : {len(registre_exemples['chat_fr_natif']):,} / {QUOTAS_CATEGORIES['chat_fr_natif']:,} (dont {nb_fr_multi} multi-tours OASST)")

# 3B. Chat EN natif : everyday-conversations (multi-tours 4-5 tours) + smoltalk en_en propre
print("Téléchargement EN natif : HuggingFaceTB/everyday-conversations-llama3.1-2k")
chemin_ev = hf_hub_download(
    repo_id=DEPOT_EVERYDAY,
    filename="data/train_sft-00000-of-00001.parquet",
    repo_type="dataset",
    token=JETON_HF,
    local_dir=str(DOSSIER_CACHE / "everyday"),
)
lignes_ev = pq.read_table(chemin_ev, columns=["messages"]).to_pylist()
rng_en = random.Random(GRAINE + 20)
rng_en.shuffle(lignes_ev)
for row in lignes_ev:
    if len(registre_exemples["chat_en_natif"]) >= 190:
        break
    ajouter_dialogue_sft("chat_en_natif", row["messages"], "en", "en", "everyday_en_natif", verifier_langue_reponse=False)

print("Complément EN natif + BCI pur : Tree-AI-lab/smoltalk-multilingue (shard 0)")
chemin_st0 = hf_hub_download(
    repo_id=DEPOT_SMOLTALK,
    filename="data/train/train-00000-of-00008.parquet",
    repo_type="dataset",
    token=JETON_HF,
    local_dir=str(DOSSIER_CACHE / "smoltalk"),
)
SOURCES_ST_SAINES = {"smol-magpie-ultra", "smol-summarize", "smol-rewrite", "openhermes-100k", "explore-instruct-rewriting", "everyday-conversations"}
table_st0 = pq.read_table(chemin_st0, columns=["messages", "source", "direction", "langue_de_reponse_attendue"])
indices_st = list(range(table_st0.num_rows))
rng_en.shuffle(indices_st)

col_m, col_s, col_d, col_c = table_st0["messages"], table_st0["source"], table_st0["direction"], table_st0["langue_de_reponse_attendue"]
cible_bci_pur = 350
nb_bci_pur = 0

for idx in tqdm(indices_st, desc="smoltalk_en_et_bci_pur", leave=False):
    if len(registre_exemples["chat_en_natif"]) >= QUOTAS_CATEGORIES["chat_en_natif"] and nb_bci_pur >= cible_bci_pur:
        break
    dr = str(col_d[idx].as_py() or "")
    if dr not in ("en_en", "bci_bci"):
        continue
    src = str(col_s[idx].as_py() or "")
    if src not in SOURCES_ST_SAINES:
        continue
    msgs = col_m[idx].as_py()
    cons = str(col_c[idx].as_py() or "").strip()
    if not isinstance(msgs, list) or len(msgs) < 2:
        continue
    msgs_clean = []
    for m in msgs:
        c = str(m.get("content") or "")
        if m.get("role") == "user" and cons:
            c = c.replace(cons, "").strip()
        msgs_clean.append({"role": m.get("role"), "content": c})

    if dr == "en_en" and len(registre_exemples["chat_en_natif"]) < QUOTAS_CATEGORIES["chat_en_natif"]:
        ajouter_dialogue_sft("chat_en_natif", msgs_clean, "en", "en", f"smoltalk_en_{src}", verifier_langue_reponse=False)
    elif dr == "bci_bci" and nb_bci_pur < cible_bci_pur:
        if ajouter_dialogue_sft("chat_bci_dyu", msgs_clean, "bci", "bci", f"smoltalk_bci_{src}", verifier_langue_reponse=True):
            nb_bci_pur += 1

print(f"[OK] Chat EN natif : {len(registre_exemples['chat_en_natif']):,} | Chat BCI pur : {nb_bci_pur:,}")

## 4. Bloc 3 (suite) : Intentions Assistant Dioula/Baoulé + Cours Dioula Propres (`cosmopedia-v2`) + Multi-Tours Trilingues Nettoyés + Traduction 1:1

In [ ]:
# 4A-0. Intentions d'assistant naturelles en Dioula pur & Baoulé pur (alignement comportemental LIMA pur :
# répond directement à la question/salutation sans JAMAIS réciter sa biographie hors-sujet !)
DIALOGUES_ASSISTANT_DYU_BCI_CURATES = [
    # --- Dioula pur (salutations simples, questions courtes, demandes d'explication, conseils, reformulation, calcul) ---
    ("I ni sɔgɔma", "Nse, i ni sɔgɔma ! I ka kɛnɛ wa ? N bɛ se ka i dɛmɛ cogo di bi ?", "dyu"),
    ("i ni tile", "Nse, i ni tile ! Fɔ n’ ye i ka ɲiningali ye mun ye bi.", "dyu"),
    ("i ni wula", "Nse, i ni wula ! I b’a fɛ an ka kuma mun kan bi wula fɛ ?", "dyu"),
    ("I ka kɛnɛ wa ?", "N ka kɛnɛ kosɔbɛ, barika ! E do, i ka kɛnɛ wa ? N bɛ se ka i dɛmɛ mun na ?", "dyu"),
    ("i ni ce kosɔbɛ", "I ni ce fana ! Ni ɲiningali wɛrɛ b’i bolo, i bɛ se k’a fɔ n’ ye tuma bɛɛ.", "dyu"),
    ("N b’a fɛ ka ɲiningali dɔ kɛ i la.", "Ayiwa, n bɛ i lamɛnna ! I ka ɲiningali kɛ.", "dyu"),
    ("n b'a fɛ i ka kuma n ye sɛnɛkɛ kan", "Sɛnɛkɛ ye baara jɔnjɔn ye min bɛ dumuni ni wari di mɔgɔw ma. Kɔdiwari jamana na, sɛnɛkɛlaw bɛ malo, kaba, ku, banaan, kakawɔ ani kafe sɛnɛ.", "dyu"),
    ("mɔbili mɔtɛri bɛ baara kɛ cogo di", "Mɔbili mɔtɛri bɛ esansi wala gazuwali jeni a kɔnɔ walisa ka fanga labɔ. O fanga bɛ nɛgɛw lamaga, o kɔ fɛ o bɛ mɔbili sɛnw munu ka mɔbili taama.", "dyu"),
    ("mun na ji sɛniman ka kan kosɔbɛ ?", "Ji sɛniman bɛ adamaden tanga kɔnɔdimi ni banaw ma, a bɛ fari lafiya ani a bɛ kɛnɛya sabati loon o loon.", "dyu"),
    ("kɛnɛya ladili saba di n’ ma", "Kɛnɛya ladili saba filɛ : 1) Ji sɛniman min loon o loon, 2) Dumuni suguya caman dumu (nakɔfɛnw ni yiridenw), 3) Sunɔgɔ ka ɲɛ su fɛ ani ka farikoloɲɛnajɛ kɛ.", "dyu"),
    ("fɛɛn jumanw bɛ kɛ ka malo tobi ka ɲɛ ?", "Walisa ka malo tobi ka ɲɛ : a ko fɔlɔ ni ji sɛniman ye, ji hakɛ bɛnnen kɛ a kan, a datugu ka a tobi ta dɔgɔman kan, ani a to a ka lafiya miniti duuru sanni i k’a tila.", "dyu"),
    ("baara kɛli ɲɔgɔn fɛ nafa ye mun ye ?", "Ni mɔgɔw bɛ baara kɛ ɲɔgɔn fɛ, baara bɛ kɛ joona, bɛɛ b’a ka dɔnniya di, ani koo gwɛlɛnw bɛ ɲɛnabɔ ka nɔgɔya.", "dyu"),
    ("kalan nafa ye mun ye denmisɛnw ye ?", "Kalan bɛ dɔnniya di denmisɛnw ma : u bɛ sɛbɛnni, kalanni ani jatebɔ dege walisa ka baara ɲuman sɔrɔ ani ka u ka jamana yiriwa.", "dyu"),
    ("25 fara 17 kan, o bɛ kɛ joli ye ?", "25 + 17 bɛ kɛ **42** ye (20 + 10 = 30 ani 5 + 7 = 12, o bɛɛ lajɛlen ye 42).", "dyu"),
    ("tɛlɛfɔni batiri bɛ se ka mara ka mɛɛn cogo di ?", "Walisa tɛlɛfɔni batiri ka mɛɛn : 1) A yeelen dɔgɔya, 2) Porogaramuw datugu ni i tɛ baara kɛ n’u ye, 3) Wi-Fi ni Bluetooth datugu ni i mako t’u la.", "dyu"),
    ("sanjikalo la, sɛnɛkɛlaw bɛ mun kɛ ?", "Sanjikalo la, sɛnɛkɛlaw bɛ u ka forow labɛn, ka siw dan (i n’a fɔ malo, kaba wala tiga), ani ka bin juguw bɔ foro kɔnɔ walisa sumankisɛw ka fali ka ɲɛ.", "dyu"),
    ("i bɛ se ka nin kuma in kɛ surun ye : denbaya ka kan ka bɛn ani ka ɲɔgɔn dɛmɛ tuma bɛɛ ?", "A surunya filɛ : **Bɛn ni dɛmɛɲɔgɔnya bɛ denbaya sabati.**", "dyu"),
    ("yiriw turuli nafa ye mun ye an ka duguw la ?", "Yiriw bɛ suma di, u bɛ fiɲɛ sɛniya, u bɛ dugukolo tanga sanji fanga ma, ani u bɛ yiridenw di mɔgɔw ma.", "dyu"),
    # --- Baoulé pur (salutations simples, questions courtes, explications, conseils, calcul) ---
    ("Aɲiho", "Aɲiho ! A ti juejue ? Ngue sa yɛ n kwla uka wɔ su andɛ ɔ ?", "bci"),
    ("awossi'n o", "Awossi'n o ! A ti juejue ? Kan like nga a kunndɛ kɛ e koko su yalɛ nnɔsua nga nun’n kle min.", "bci"),
    ("a ti juejue ?", "N ti juejue kpa, mo ! Yɛ wɔ li, a ti juejue ? Ngue sa yɛ n kwla yo man wɔ andɛ ɔ ?", "bci"),
    ("mo kpa", "A yo ɔ mo ! Sɛ a le kosan uflɛ’n, usa min blɛ kwlaa nun.", "bci"),
    ("n kunndɛ kɛ n usa wɔ kosan kun", "Ɛɛn, n su tie wɔ ! Usa ɔ kosan’n.", "bci"),
    ("kan fie dilɛ’n i ndɛ kle min", "Fie dilɛ’n ti junman cinnjin kpa Kɔti Divwari lɔ. Fie nun yɛ be ɲan aliɛ kɛ avie, duɔ, banda nin maniɔku sa ɔ, yɛ kakao nin kafe be man sika.", "bci"),
    ("loto i mɔtɛri’n di junman sɛ ?", "Loto i mɔtɛri’n yra esansi annzɛ gazuwali i klun lɔ naan w’a yi wunmiɛn. Wunmiɛn sɔ’n yɛ ɔ kɛnkɛn ninnge mun naan loto’n i ja’m b’a kpɛnkpɛn b’a kɔ ɲrun ɔn.", "bci"),
    ("ngue ti yɛ nzue sanwun’n ti cinnjin kpa ɔ ?", "Nzue sanwun’n sasa e wunnɛn’n tukpacɛ mun be lika, ɔ uka e naan y’a tran juejue yɛ y’a di junman kpa cɛn kwlaa.", "bci"),
    ("man min juejue tranlɛ afɔtuɛ nsan", "Juejue tranlɛ afɔtuɛ nsan yɛ : 1) Nɔn nzue sanwun kpanngban, 2) Di aliɛ kpa (mma nin nɲanman), 3) Lafi kpa kɔnguɛ yɛ cici ɔ wunnɛn’n.", "bci"),
    ("sɛ e fa 14 e uka 9 su’n, ɔ yo nɲɛ ?", "14 + 9 yo **23**.", "bci"),
    ("wafa sɛ yɛ be tɔn avie kpa ɔ ?", "Kɛ á tɔ́n avie’n, wunnzin i kpa nzue sanwun nun, fa nzue ng’ɔ nin i fata’n gua su, kata su sin kaan su, kpɛkun yaci i lɛ miniti nnun ka naan w’a yi.", "bci"),
    ("ngue ti yɛ suklu dilɛ’n ti kpa ba’m be ti ɔ ?", "Suklu dilɛ’n kle ba’m be fluwa kannganlɛ, fluwa klɛlɛ nin akɔnta bulɛ, naan be kwla ɲan junman kpa ainman.", "bci"),
    ("waka talɛ’n i su mmlusuɛ’n yɛle benin ?", "Waka’m be man fɔun, be yo maan aunmuan’n yo sanwun, be sasa asiɛ’n nzue’n i lika, yɛ be man e mma.", "bci"),
    ("a kwla yo ndɛ nga i tika : awlobofuɛ’m be wun ukalɛ’n man fɔun nin wunmiɛn ?", "I tika’n yɛ : **Awlobo nun anuanzɛ’n man wunmiɛn.**", "bci"),
]
nb_asst_cur = 0
for u_cur, a_cur, lg_cur in DIALOGUES_ASSISTANT_DYU_BCI_CURATES:
    if ajouter_dialogue_sft(
        "chat_bci_dyu",
        [{"role": "user", "content": u_cur}, {"role": "assistant", "content": a_cur}],
        lg_cur, lg_cur, "assistant_dyu_bci_curate",
        verifier_langue_reponse=True,
        appliquer_clavier=False,
    ):
        nb_asst_cur += 1
print(f"[OK] Dialogues comportement assistant pur (DYU/BCI) : {nb_asst_cur:,}")

# 4A. +340 cours & explications PROPRES en Dioula pur (filtrés anti-Sapitiri, anti-'/', anti-'Aw ni ce' bienvenue)
print("Streaming éducatif Dioula pur (filtrage strict V4.5) : cosmopedia-v2/smollm_cosmopedia-v2_001_dyu.jsonl")
URL_COSMO_DYU = f"https://huggingface.co/datasets/{DEPOT_SMOLLM}/resolve/main/cosmopedia-v2/smollm_cosmopedia-v2_001_dyu.jsonl"

# 12 formulations conversationnelles naturelles variées (finit les 5 templates rigides + style clavier/ASR activé !)
TEMPLATES_Q_DYU_COURS = [
    "{sujet} ye mun ye ?",
    "{sujet} ɲɛfɔ n’ ye ka ɲɛ.",
    "N b’a fɛ ka {sujet} faamu.",
    "Kuma n’ ye {sujet} kan.",
    "I bɛ se ka {sujet} ɲɛfɔ n’ ye wa ?",
    "I bɛ se ka {sujet} faamuya n’ ye cogo di ?",
    "N b’a fɛ ka dɔ lɔn {sujet} koo la.",
    "Fɔ n’ ye fɛɛn jɔnjɔnw {sujet} kan.",
    "Kalansen surun dɔ kɛ n’ ye {sujet} kan.",
    "Dɔ fɔ n’ ye {sujet} kan.",
    "Mun lo bɛ se ka fɔ {sujet} kan ?",
    "N bɛ ɲiningali kɛ {sujet} kan, a ɲɛfɔ n’ ye.",
]

REGEX_COSMO_BRUIT_TITRE = re.compile(
    r"(?:sapitiri|chapitre|kalansen\s+yɔrɔ|/|eckankar|mahanta|vision\s+quests|<|>|\{|\}|\[|\]|\||=|\d{3,})",
    re.IGNORECASE,
)
REGEX_COSMO_BRUIT_CORPS = re.compile(
    r"(?:^|\n)(?:I+\.|V\.|Sapitiri|Kalansen\s+fɔlɔ|Daminɛ\s+kuma)|\bAw\s+ni\s+ce\s+an\s+ka\b|\*\*|##",
    re.IGNORECASE,
)

rng_cosmo = random.Random(GRAINE + 35)
banque_dyu_cours: list[tuple[str, str]] = []  # (question_dyu, reponse_longue_dyu)
cible_dyu_pur = 340
nb_dyu_pur = 0


def couper_proprement_paragraphes(corps: str, max_car: int = 1100) -> str:
    """Coupe proprement à la fin d'une phrase ou d'un paragraphe (jamais au milieu d'un mot)."""
    corps = nettoyer_texte(corps)
    if len(corps) <= max_car:
        return corps
    morceau = corps[:max_car]
    dernier_point = max(morceau.rfind("."), morceau.rfind("!"), morceau.rfind("?"))
    if dernier_point >= 320:
        return morceau[: dernier_point + 1].strip()
    return morceau.rsplit(" ", 1)[0].strip() + "."


lignes_lues_cosmo = 0
with requests.get(URL_COSMO_DYU, headers={"Authorization": f"Bearer {JETON_HF}"}, stream=True, timeout=60) as resp_cosmo:
    resp_cosmo.raise_for_status()
    for ligne_str in resp_cosmo.iter_lines(decode_unicode=True):
        if nb_dyu_pur >= cible_dyu_pur or lignes_lues_cosmo >= 25_000:
            break
        if not ligne_str:
            continue
        lignes_lues_cosmo += 1
        try:
            obj = json.loads(ligne_str)
        except json.JSONDecodeError:
            continue
        texte_brut = nettoyer_texte(str(obj.get("text") or ""))
        if len(texte_brut) < 420 or "```" in texte_brut or "http" in texte_brut.lower():
            continue
        lignes_txt = [l.strip() for l in texte_brut.split("\n") if l.strip()]
        if len(lignes_txt) < 2:
            continue
        titre = lignes_txt[0]
        if REGEX_COSMO_BRUIT_TITRE.search(titre):
            rejets["cosmo_titre_bruit"] += 1
            continue
        titre_propre = re.sub(
            r"^(?:Kalansen|Barokun|Kuntilenna|Sɛbɛn\s+tɔgɔ)\s*:\s*", "", titre, flags=re.IGNORECASE
        ).strip(' .#*_-«»"“”')
        mots_titre = titre_propre.split()
        # Titre court et clair (3 à 8 mots max, pas de titre de chapitre géant ni de phrase interrogative)
        if not (14 <= len(titre_propre) <= 68) or not (3 <= len(mots_titre) <= 8) or "?" in titre_propre:
            rejets["cosmo_titre_long_ou_invalide"] += 1
            continue
        corps = "\n\n".join(lignes_txt[1:])
        reponse_dyu = couper_proprement_paragraphes(corps, max_car=1050)
        if len(reponse_dyu) < 340 or REGEX_COSMO_BRUIT_CORPS.search(reponse_dyu):
            rejets["cosmo_corps_bruit_ou_aw_ni_ce"] += 1
            continue
        if detecter_motif_rejet(titre_propre, langue_attendue="dyu") is not None:
            continue
        if detecter_motif_rejet(reponse_dyu, langue_attendue="dyu") is not None:
            continue
        question_dyu = rng_cosmo.choice(TEMPLATES_Q_DYU_COURS).format(sujet=titre_propre)
        if ajouter_dialogue_sft(
            "chat_bci_dyu",
            [{"role": "user", "content": question_dyu}, {"role": "assistant", "content": reponse_dyu}],
            "dyu", "dyu", "cosmopedia_dyu_propre_v45", verifier_langue_reponse=True, appliquer_clavier=True,
        ):
            nb_dyu_pur += 1
            if len(banque_dyu_cours) < 120:
                banque_dyu_cours.append((question_dyu, reponse_dyu))

if nb_dyu_pur < 180:
    raise ErreurDonneeInvalide(f"[ERREUR] Trop peu de cours Dioula propres extraits de Cosmopedia-v2 : {nb_dyu_pur} (< 180)")
print(f"[OK] Cours Dioula pur propres V4.5 (Cosmopedia-v2) : {nb_dyu_pur:,} extraits sur {lignes_lues_cosmo:,} lignes streamées")

# 4B. Multi-tour Trilingue (4 tours complets, suffixes 100 % retirés, diversité d'amorces + vérification langue par tour)
print("Téléchargement : Tree-AI-lab/kouman-conversations-trilingues-longues-25k")
chemin_tri = hf_hub_download(
    repo_id=DEPOT_TRILINGUES,
    filename="train_trilingual_25k.jsonl",
    repo_type="dataset",
    token=JETON_HF,
    local_dir=str(DOSSIER_CACHE / "trilingues"),
)
rng_tri = random.Random(GRAINE + 30)
with open(chemin_tri, "r", encoding="utf-8") as fh:
    lignes_tri = [json.loads(l) for l in fh if l.strip()]
rng_tri.shuffle(lignes_tri)

sigs_a0: set[str] = set()
compteur_amorces_u0: Counter = Counter()
nb_tri_aj = 0
for d in lignes_tri:
    if len(registre_exemples["chat_bci_dyu"]) >= QUOTAS_CATEGORIES["chat_bci_dyu"]:
        break
    msgs = d.get("messages") or []
    if len(msgs) < 6:
        continue
    a0 = nettoyer_texte(msgs[1].get("content", ""))
    sig = re.sub(r"\s+", " ", re.sub(r"\d+", "N", a0.lower()))[:90]
    if sig in sigs_a0:
        rejets["tri_clone_a0"] += 1
        continue
    msgs_p = []
    dialogue_sain = True
    for m in msgs[:8]:  # 4 tours complets (8 messages)
        r, c = str(m.get("role") or ""), nettoyer_texte(str(m.get("content") or ""))
        if r == "user":
            c = retirer_suffixes_forces(c)
            if REGEX_RESIDU_SUFFIXE_INTERDIT.search(c):
                rejets["tri_suffixe_residu"] += 1
                dialogue_sain = False
                break
        elif r == "assistant":
            # Contrôle de langue individuel par tour (au lieu de verifier_langue_reponse=False aveugle)
            lg_tour = "bci" if REGEX_CAR_BCI_DYU.search(c) else ("fr" if ratio_correspondances(REGEX_MOTS_FR, c) >= 0.12 else None)
            if detecter_motif_rejet(c, langue_attendue=lg_tour) is not None:
                rejets["tri_tour_asst_invalide"] += 1
                dialogue_sain = False
                break
        msgs_p.append({"role": r, "content": c})
    if not dialogue_sain or len(msgs_p) < 6:
        continue
    # Plafonne à 3 dialogues max par amorce répétitive (évite les clones "Je prépare une fiche de calcul mental...")
    cle_amorce = re.sub(r"\s+", " ", msgs_p[0]["content"].lower())[:36]
    if compteur_amorces_u0[cle_amorce] >= 3:
        rejets["tri_amorce_repetitive"] += 1
        continue
    if ajouter_dialogue_sft("chat_bci_dyu", msgs_p, "multi", "multi", "trilingues_4tours_propre_v45", verifier_langue_reponse=False):
        sigs_a0.add(sig)
        compteur_amorces_u0[cle_amorce] += 1
        nb_tri_aj += 1

print(f"[OK] Chat BCI+DYU total : {len(registre_exemples['chat_bci_dyu']):,} (BCI={nb_bci_pur} | Assistant curaté={nb_asst_cur} | DYU cours={nb_dyu_pur} | Trilingues 4-tours={nb_tri_aj})")

# 4C. Traduction directe 1 phrase <-> 1 phrase (1 450 = identique V4.2) + collecte banque courte bilingue
NOMS_FR = {"fr": "français", "en": "anglais", "bci": "baoulé", "dyu": "dioula"}
NOMS_EN = {"fr": "French", "en": "English", "bci": "Baoulé", "dyu": "Dioula"}
TEMPLATES_TRAD_FR = [
    "Traduis en {cible} : « {texte} »",
    "Traduis cette phrase en {cible} : {texte}",
]
TEMPLATES_TRAD_EN = [
    'Translate to {cible}: "{texte}"',
    "Translate this sentence into {cible}: {texte}",
]

CONFIGS_SMOLLM = [
    (
        "fineweb_dyu_003",
        {
            "dyu": "fineweb-edu-dedup/smollm_fineweb-edu-dedup_003_dyu.parquet",
            "fr": "fineweb-edu-dedup/smollm_fineweb-edu-dedup_003_fr.parquet",
            "en": "fineweb-edu-dedup/smollm_fineweb-edu-dedup_003_en.parquet",
        },
        [("fr", "dyu"), ("dyu", "fr"), ("en", "dyu"), ("dyu", "en")],
    ),
    (
        "fineweb_bci_006",
        {
            "bci": "fineweb-edu-dedup/smollm_fineweb-edu-dedup_006_bci-00000.parquet",
            "fr": "fineweb-edu-dedup/smollm_fineweb-edu-dedup_006_fr-00000.parquet",
            "en": "fineweb-edu-dedup/smollm_fineweb-edu-dedup_006_en-00000.parquet",
        },
        [("fr", "bci"), ("bci", "fr"), ("en", "bci"), ("bci", "en")],
    ),
    (
        "cosmopedia_bci_005",
        {
            "bci": "cosmopedia-v2/smollm_cosmopedia-v2_005_bci-00000.parquet",
            "fr": "cosmopedia-v2/smollm_cosmopedia-v2_005_fr-00000.parquet",
            "en": "cosmopedia-v2/smollm_cosmopedia-v2_005_en-00000.parquet",
        },
        [("fr", "bci"), ("bci", "fr"), ("en", "bci"), ("bci", "en")],
    ),
]


def cle_base_smollm(doc_id: str) -> str:
    return re.sub(r"_(?:fr|en|bci|dyu|nch)$", "", str(doc_id).strip())


def decouper_phrases_courtes(texte: str) -> list[str]:
    bruts = re.split(r"(?<=[.!?])\s+", nettoyer_texte(texte))
    return [p.strip() for p in bruts if 22 <= len(p.strip()) <= 155 and "\n" not in p.strip()]


def phrase_propre_sans_bruit(texte: str, langue: str) -> bool:
    if detecter_motif_rejet(texte, langue_attendue=langue) is not None:
        return False
    if re.search(r"def\s+\w+\(|#include|class\s+\w+:|</?[a-z]+>", texte):
        return False
    return True


rng_sm = random.Random(GRAINE + 40)
compteurs_sens = Counter()
banque_courte_bilingue: dict[str, list[tuple[str, str]]] = {"bci": [], "dyu": []}

# 20 paires vérifiées du vocabulaire quotidien ivoirien (riz=malo/avie, sœur, grands-parents, village, matin, école, famille)
# Garantit la précision lexicale sur les mots usuels (jamais 'mugu' pour 'riz') — 100 % distinct des phrases OOD !
TRADUCTIONS_QUOTIDIENNES_VERIFIEES = [
    ("fr", "dyu", "Le riz chaud est prêt sur la table.", "Malo kalaman bɛɛ labɛnna tabali kan."),
    ("fr", "dyu", "Ma sœur achète du riz et des légumes au marché.", "N balimamuso bɛ malo ni nakɔfɛnw san lɔgɔfiyɛ la."),
    ("fr", "dyu", "Mon frère prépare le repas du soir pour les invités.", "N balimakɛ bɛ surafana tobi lonanw kama."),
    ("fr", "dyu", "Les enfants saluent leurs grands-parents chaque matin.", "Denmisɛnw bɛ u mɔkɛw ni u mɔmusow fo sɔgɔma o sɔgɔma."),
    ("fr", "dyu", "Demain matin, nous irons travailler au champ ensemble.", "Sini sɔgɔma, an bɛna taga baara kɛ foro la ɲɔgɔn fɛ."),
    ("fr", "dyu", "La famille partage un bon repas dans la cour du village.", "Denbaya bɛ dumuni duman kɛ ɲɔgɔn fɛ dugu lu kɔnɔ."),
    ("fr", "dyu", "Ma mère fait bouillir l'eau avant de préparer la sauce.", "N ba bɛ ji wuli sanni a ka naani tobi."),
    ("fr", "dyu", "Mon père revient du village avec des mangues mûres.", "N fa bɛ segi ka bɔ dugu la ni mango mɔninw ye."),
    ("dyu", "fr", "Malo ni tiga naani ka di denbaya bɛɛ ye.", "Le riz à la sauce arachide plaît à toute la famille."),
    ("dyu", "fr", "Sini sɔgɔma, n balimamuso bɛna taga kalanso la.", "Demain matin, ma sœur ira à l'école."),
    ("fr", "bci", "Le riz chaud est prêt pour toute la famille.", "Avie nga ɔ ti wɛɛn’n w’a yo kpa awlobo wunmuan’n ti."),
    ("fr", "bci", "Demain matin, mon petit frère ira au marché avec notre mère.", "Aliemaa nglɛmun, min niaan bian kaan’n ɔ́ kɔ́ gua su nin e nin."),
    ("fr", "bci", "Nous allons saluer nos grands-parents au village ce dimanche.", "E su kɔ yo e nannan’m be like klɔ lɔ Mɔnnɛ nga nun."),
    ("fr", "bci", "Ma sœur prépare du riz et du poisson pour le dîner.", "Min niaan bla’n su tɔn avie nin jue nnɔsua aliɛ’n ti."),
    ("fr", "bci", "Les enfants aiment écouter les histoires des anciens au village.", "Ba’m be klo kɛ bé tíe sran kpɛnngbɛn’m be ndɛ mun klɔ lɔ."),
    ("fr", "bci", "L'eau fraîche du puits est bonne à boire après le travail.", "Nzue fɔun ng’ɔ fin gbo nun’n ɔ ti kpa nɔnlɛ kɛ be wie junman di’n."),
    ("bci", "fr", "Aliemaa nglɛmun, e su kɔ fie su kɛ é kó di junman.", "Demain matin, nous allons au champ pour travailler."),
    ("bci", "fr", "Min niaan bla’n tɔnnin avie kpa awlobo’n ti.", "Ma sœur a préparé du bon riz pour la famille."),
    ("en", "dyu", "Tomorrow morning I will visit my grandparents in the village.", "Sini sɔgɔma, n bɛna taga n mɔkɛw ni n mɔmusow lajɛ dugu la."),
    ("en", "bci", "My sister is cooking rice for the whole family tonight.", "Min niaan bla’n su tɔn avie awlobo wunmuan’n ti kɔnguɛ nga nun."),
]
for src_v, dst_v, txt_s, txt_d in TRADUCTIONS_QUOTIDIENNES_VERIFIEES:
    if "en" in (src_v, dst_v):
        q_v = rng_sm.choice(TEMPLATES_TRAD_EN).format(cible=NOMS_EN[dst_v], texte=txt_s)
        lg_cv = "en"
    else:
        q_v = rng_sm.choice(TEMPLATES_TRAD_FR).format(cible=NOMS_FR[dst_v], texte=txt_s)
        lg_cv = "fr"
    if ajouter_dialogue_sft(
        "traduction_directe",
        [{"role": "user", "content": q_v}, {"role": "assistant", "content": txt_d}],
        langue_consigne=lg_cv,
        langue_reponse=dst_v,
        source=f"trad_quotidien_ci_{src_v}_{dst_v}",
        verifier_langue_reponse=True,
    ):
        compteurs_sens[(src_v, dst_v)] += 1

for nom_grp, fichiers_lang, sens_list in CONFIGS_SMOLLM:
    besoin_grp_trad = any(compteurs_sens[s] < QUOTAS_TRAD_DIRECTE[s] for s in sens_list)
    lg_loc_grp = "dyu" if "dyu" in fichiers_lang else "bci"
    besoin_grp_banque = len(banque_courte_bilingue[lg_loc_grp]) < 750
    if not (besoin_grp_trad or besoin_grp_banque):
        continue
    print(f"Alignement 1:1 SmolLM : {nom_grp}")
    tables_lg: dict[str, dict[str, str]] = {}
    for lg, fic in fichiers_lang.items():
        chemin_p = hf_hub_download(
            repo_id=DEPOT_SMOLLM,
            filename=fic,
            repo_type="dataset",
            token=JETON_HF,
            local_dir=str(DOSSIER_CACHE / "smollm"),
        )
        t = pq.read_table(chemin_p, columns=["id", "text"])
        tables_lg[lg] = {cle_base_smollm(i): txt for i, txt in zip(t["id"].to_pylist(), t["text"].to_pylist()) if isinstance(txt, str) and len(txt) >= 25}

    cles = sorted(set.intersection(*(set(d.keys()) for d in tables_lg.values())))
    rng_sm.shuffle(cles)

    for cle in cles:
        besoin_trad = any(compteurs_sens[s] < QUOTAS_TRAD_DIRECTE[s] for s in sens_list)
        lg_loc = "dyu" if "dyu" in tables_lg else "bci"
        besoin_banque = len(banque_courte_bilingue[lg_loc]) < 750
        if not besoin_trad and not besoin_banque:
            break

        if besoin_banque:
            p_fr = decouper_phrases_courtes(tables_lg["fr"][cle])
            p_loc = decouper_phrases_courtes(tables_lg[lg_loc][cle])
            if p_fr and len(p_fr) == len(p_loc):
                s_fr, s_loc = nettoyer_texte(p_fr[0]), nettoyer_texte(p_loc[0])
                if (
                    20 <= len(s_fr) <= 85
                    and 20 <= len(s_loc) <= 95
                    and not re.search(r"\d|[()\[\]{}/\\]|[A-Z]{3,}", s_fr + s_loc)
                    and phrase_propre_sans_bruit(s_fr, "fr")
                    and phrase_propre_sans_bruit(s_loc, lg_loc)
                ):
                    banque_courte_bilingue[lg_loc].append((s_fr, s_loc))

        for src_lg, dst_lg in sens_list:
            if compteurs_sens[(src_lg, dst_lg)] >= QUOTAS_TRAD_DIRECTE[(src_lg, dst_lg)]:
                continue
            ph_s = decouper_phrases_courtes(tables_lg[src_lg][cle])
            ph_d = decouper_phrases_courtes(tables_lg[dst_lg][cle])
            if not ph_s or len(ph_s) != len(ph_d):
                continue
            k = rng_sm.randrange(len(ph_s))
            s_s, s_d = ph_s[k], ph_d[k]
            if not (0.45 <= len(s_d) / max(len(s_s), 1) <= 2.2):
                continue
            if not (phrase_propre_sans_bruit(s_s, src_lg) and phrase_propre_sans_bruit(s_d, dst_lg)):
                continue
            if "en" in (src_lg, dst_lg):
                q = rng_sm.choice(TEMPLATES_TRAD_EN).format(cible=NOMS_EN[dst_lg], texte=s_s)
                lg_c = "en"
            else:
                q = rng_sm.choice(TEMPLATES_TRAD_FR).format(cible=NOMS_FR[dst_lg], texte=s_s)
                lg_c = "fr"
            if ajouter_dialogue_sft(
                "traduction_directe",
                [{"role": "user", "content": q}, {"role": "assistant", "content": s_d}],
                langue_consigne=lg_c,
                langue_reponse=dst_lg,
                source=f"smollm_1a1_{src_lg}_{dst_lg}",
                verifier_langue_reponse=True,
            ):
                compteurs_sens[(src_lg, dst_lg)] += 1

# Complète chat_croise_explicite (jusqu'à 100) avec des réponses doubles propres (FR puis BCI / FR puis DYU)
for lg_dbl, nom_dbl, titre_dbl in [("bci", "baoulé", "Baoulé"), ("dyu", "dioula", "Dioula")]:
    cible_aj = 21
    nb_aj_d = 0
    for s_fr, s_loc in banque_courte_bilingue[lg_dbl][500:]:
        if nb_aj_d >= cible_aj or len(registre_exemples["chat_croise_explicite"]) >= QUOTAS_CATEGORIES["chat_croise_explicite"]:
            break
        q_dbl = rng_sm.choice([
            f"Écris cette phrase en français puis en {nom_dbl} : {s_fr}",
            f"Formule cette idée en français et en {nom_dbl} : {s_fr}",
            f"Donne-moi en français puis en {nom_dbl} : {s_fr}",
        ])
        a_dbl = f"**Français :** {s_fr}\n\n**{titre_dbl} :** {s_loc}"
        if ajouter_dialogue_sft(
            "chat_croise_explicite",
            [{"role": "user", "content": q_dbl}, {"role": "assistant", "content": a_dbl}],
            langue_consigne="fr",
            langue_reponse="multi",
            source=f"double_langue_fr_{lg_dbl}",
            verifier_langue_reponse=False,
        ):
            nb_aj_d += 1

print(f"[OK] Chat croisé & doubles langues (100 % propre) : {len(registre_exemples['chat_croise_explicite']):,} / {QUOTAS_CATEGORIES['chat_croise_explicite']}")
print(f"[OK] Traductions directes 1:1 : {len(registre_exemples['traduction_directe']):,} | {dict(compteurs_sens)}")
print(f"Banque courte bilingue collectée : BCI={len(banque_courte_bilingue['bci'])}, DYU={len(banque_courte_bilingue['dyu'])}")

## 5. Bloc 4 (`chat_multitour_relais` 300) + Bloc 6 (`melange_bilingue` 730) + Bloc 7 (`identite_legere` 220)

In [ ]:
# 5A. Bloc Multi-Tour & Changements de Sujet (300 dialogues de 4 à 5 tours : relances courtes, cours après salutation, bascules)
rng_mt = random.Random(GRAINE + 45)

DIALOGUES_MULTITOUR_CURATES = [
    # 1. Relances courtes de salutations en dioula sans jamais déclencher "Au revoir" au tour 4
    [
        {"role": "user", "content": "comment on dit bonjour en dioula"},
        {"role": "assistant", "content": "En dioula, « bonjour » se dit selon le moment de la journée : **I ni sɔgɔma** (le matin), **I ni tile** (en journée) et **I ni wula** (le soir)."},
        {"role": "user", "content": "et comment on dit merci ?"},
        {"role": "assistant", "content": "En dioula, « merci » se dit : **I ni ce** (à une personne) ou **Aw ni ce** (à plusieurs personnes). On utilise aussi **Barika**."},
        {"role": "user", "content": "et bonjour le matin ?"},
        {"role": "assistant", "content": "Le matin, on dit : **I ni sɔgɔma** (et la personne répond généralement **Nse** ou **Mba**, puis **I ni sɔgɔma**)."},
        {"role": "user", "content": "et en baoulé comment on dit bonjour et merci ?"},
        {"role": "assistant", "content": "En baoulé, « bonjour » se dit **Aɲiho**, « bienvenue » se dit **Akwaba**, et « merci » se dit **Mo** ou **A yo ɔ mo**."},
    ],
    # 2. Dialogue Dioula complet : salutation -> demande d'apprentissage -> vrai cours détaillé (zéro réponse timide !)
    [
        {"role": "user", "content": "I ni sɔgɔma"},
        {"role": "assistant", "content": "Nse, i ni sɔgɔma ! I ka kɛnɛ wa ? N bɛ se ka i dɛmɛ cogo di bi ?"},
        {"role": "user", "content": "N ka kɛnɛ kosɔbɛ. N b’a fɛ ka kalan kɛ bi."},
        {"role": "assistant", "content": "O ka ɲi kosɔbɛ ! I b’a fɛ an ka daminɛ kalansen juman na bi : jatebɔ (calcul), dɔgɔyali, tariku, wala kanw kalanni ?"},
        {"role": "user", "content": "Faraɲɔgɔnkan (addition) ɲɛfɔ n’ ye."},
        {"role": "assistant", "content": "Ayiwa, an ka daminɛ faraɲɔgɔnkan (addition) na !\n\nFaraɲɔgɔnkan kɔrɔ ye ka jatew fara ɲɔgɔn kan walisa ka u bɛɛ lajɛlen sɔrɔ. Misali la :\n1. Ni mango **3** b’i bolo ani i bɛ **2** wɛrɛ sɔrɔ, o bɛ kɛ `3 + 2 = 5` (mango duuru).\n2. Ni i bɛ `15 + 12` jate, i bɛ tanw fara ɲɔgɔn kan (`10 + 10 = 20`) ani kelenw fara ɲɔgɔn kan (`5 + 2 = 7`), o bɛ di `27`.\n\nI b’a fɛ n ka ɲiningali kelen di i ma i k’a lajɛ wa ?"},
        {"role": "user", "content": "Ɔwɔ, di n’ ma kelen !"},
        {"role": "assistant", "content": "Ayiwa, filɛ i ka ɲiningali fɔlɔ : ni malo kilo **14** bɛ butiki la ani u bɛ kilo **11** wɛrɛ fara a kan, malo kilo joli lo bɛ sɔrɔ a bɛɛ lajɛlen na ?"},
    ],
    # 3. Dialogue Baoulé complet : salutation -> demande d'apprentissage -> explication développée
    [
        {"role": "user", "content": "Aɲiho"},
        {"role": "assistant", "content": "Aɲiho ! ?A ti juejue ? ?Ngue sa yɛ n kwla uka wɔ su andɛ ɔ ?"},
        {"role": "user", "content": "N ti juejue. N kunndɛ kɛ a kle min kɛ be bu akɔnta’n."},
        {"role": "assistant", "content": "Ɔ ti kpa dan ! E kwla bo i bo nin like fa uka su’n (addition).\n\nKɛ e fa like kun e uka uflɛ su’n, e ka be kwlaa be nuan. I wie yɛle kɛ :\n1. Sɛ a le kɔkɔti **4** naan a to **3** uflɛ’n, `4 + 3 = 7` (kɔkɔti nso).\n2. Sɛ e bu `20 + 15`, ɔ yo `35`.\n\n?A kunndɛ kɛ n man wɔ akɔnta kaan kun naan a nian sɛ a wun i wlɛ ɔ ?"},
        {"role": "user", "content": "Ɛɛn, man min kun."},
        {"role": "assistant", "content": "Kpa ! Sɛ fie difuɛ kun ti kakaɔ saki **12** cɛn kun, naan aliemaa ɔ ti saki **8** ekun’n, ?saki nɲɛ yɛ ɔ le i kwlaa ɔ ?"},
        {"role": "user", "content": "Ɔ yo 20 !"},
        {"role": "assistant", "content": "A yo ɔ mo ! `12 + 8 = 20`, ɔ ti nanwlɛ sɛsɛsɛ. ?E kɔ akɔnta uflɛ su ?"},
    ],
    # 4. Changement de sujet franc au tour 4 (pays FR -> dioula -> changement brutal vers santé/éducation en dioula)
    [
        {"role": "user", "content": "parle moi du senegal"},
        {"role": "assistant", "content": "Le Sénégal est un pays d'Afrique de l'Ouest dont la capitale est Dakar. Il est réputé pour sa tradition d'hospitalité (la teranga), sa musique, sa cuisine comme le thiéboudienne, et son dynamisme culturel."},
        {"role": "user", "content": "et quelle est sa capitale ?"},
        {"role": "assistant", "content": "La capitale du Sénégal est **Dakar**, située à l'extrême ouest du pays sur la presqu'île du Cap-Vert."},
        {"role": "user", "content": "reponds moi en dioula maintenant : pourquoi l'eau potable est importante ?"},
        {"role": "assistant", "content": "Ji sɛniman ka kan kosɔbɛ adamaden ka kɛnɛya kama. A bɛ an tanga kɔnɔdimi ni banaw caaman ma, a bɛ fari dɛmɛ ka baara kɛ ka ɲɛ, ani a bɛ denmisɛnw ka bonya sabati. O kama, an ka kan ka ji sɛniman dɔrɔn min tuma bɛɛ."},
        {"role": "user", "content": "comment on dit boire de l'eau en baoulé ?"},
        {"role": "assistant", "content": "En baoulé, « boire de l'eau » se dit : **nzue nɔnlɛ** (ou « je bois de l'eau » : **N nɔn nzue**)."},
    ],
    # 5. Relances courtes multi-langues (FR -> BCI -> DYU -> EN) sur 5 tours
    [
        {"role": "user", "content": "comment on dit bonne nuit en baoule"},
        {"role": "assistant", "content": "En baoulé, « bonne nuit » se dit : **Lafi kpa** (littéralement « dors bien ») ou **E yia nglɛmun** (« à demain matin »)."},
        {"role": "user", "content": "et en dioula ?"},
        {"role": "assistant", "content": "En dioula, « bonne nuit » se dit : **K’an bɛn sini** (à demain) ou **K’i ka su diya** / **Ala k’an sɔgɔma hɛɛrɛ la**."},
        {"role": "user", "content": "et comment on dit pardon en baoulé ?"},
        {"role": "assistant", "content": "En baoulé, « pardon / excuse-moi » se dit : **Yaci**."},
        {"role": "user", "content": "et en dioula ?"},
        {"role": "assistant", "content": "En dioula, « pardon / excuse-moi » se dit : **Hakɛto**."},
        {"role": "user", "content": "merci beaucoup !"},
        {"role": "assistant", "content": "Avec plaisir ! N'hésite pas si tu veux apprendre d'autres expressions en baoulé ou en dioula."},
    ],
    # 6. Changement de sujet franc : cuisine -> mathématiques -> traduction baoulé -> question dioula
    [
        {"role": "user", "content": "donne moi 3 conseils pour bien cuisiner le riz"},
        {"role": "assistant", "content": "Voici 3 conseils simples pour bien réussir la cuisson du riz :\n1. **Rince le riz** deux à trois fois à l'eau claire pour retirer l'excès d'amidon.\n2. **Respecte la proportion d'eau** (environ 1,5 à 2 volumes d'eau pour 1 volume de riz selon la variété).\n3. **Cuis à feu doux et à couvert** sans trop remuer, puis laisse reposer 5 minutes avant de servir."},
        {"role": "user", "content": "merci ! changeons de sujet : combien font 18 x 5 ?"},
        {"role": "assistant", "content": "`18 × 5 = 90`."},
        {"role": "user", "content": "traduis en dioula : les élèves étudient bien cette année"},
        {"role": "assistant", "content": "Kalandenw bɛ kalan kɛ ka ɲɛ ɲinan."},
        {"role": "user", "content": "et en baoulé comment on dit merci ?"},
        {"role": "assistant", "content": "En baoulé, « merci » se dit : **Mo** ou **A yo ɔ mo**."},
    ],
]

for dlg_cur in DIALOGUES_MULTITOUR_CURATES:
    ajouter_dialogue_sft(
        "chat_multitour_relais", dlg_cur, "multi", "multi", "multitour_curate",
        verifier_langue_reponse=False, appliquer_clavier=False,
    )

# Assemblage propre de dialogues 4–5 tours avec changements de sujet francs (à partir des vraies paires propres + cours Dioula)
# Apprend au modèle à rester précis au tour 4–5 sans mélanger les sujets des tours 1–3 !
pool_bci_mt = list(banque_courte_bilingue["bci"][:260])
pool_dyu_mt = list(banque_courte_bilingue["dyu"][:260])
rng_mt.shuffle(pool_bci_mt)
rng_mt.shuffle(pool_dyu_mt)

saluts_entree = [
    ("bonjour", "Bonjour ! Comment puis-je vous aider aujourd'hui ?"),
    ("salut", "Salut ! En quoi puis-je t'aider ?"),
    ("bonsoir", "Bonsoir ! Que puis-je faire pour vous ?"),
    ("I ni sɔgɔma", "Nse, i ni sɔgɔma ! N bɛ se ka i dɛmɛ cogo di bi ?"),
    ("Aɲiho", "Aɲiho ! ?Ngue sa yɛ n kwla uka wɔ su andɛ ɔ ?"),
]

idx_b, idx_d, idx_c = 0, 0, 0
while len(registre_exemples["chat_multitour_relais"]) < QUOTAS_CATEGORIES["chat_multitour_relais"]:
    if idx_b + 2 >= len(pool_bci_mt) or idx_d + 2 >= len(pool_dyu_mt):
        break
    s_u0, s_a0 = rng_mt.choice(saluts_entree)
    fr_b1, bci_1 = pool_bci_mt[idx_b]
    fr_b2, bci_2 = pool_bci_mt[idx_b + 1]
    fr_d1, dyu_1 = pool_dyu_mt[idx_d]
    fr_d2, dyu_2 = pool_dyu_mt[idx_d + 1]
    idx_b += 2
    idx_d += 2

    variante = rng_mt.randrange(3)
    if variante == 0 and idx_c < len(banque_dyu_cours):
        # 4 tours : salutation -> trad FR->BCI -> relance FR->DYU -> changement franc vers question/cours Dioula pur !
        q_c, a_c = banque_dyu_cours[idx_c]
        idx_c += 1
        msgs_mt = [
            {"role": "user", "content": s_u0},
            {"role": "assistant", "content": s_a0},
            {"role": "user", "content": rng_mt.choice([f"traduis en baoulé : {fr_b1}", f"comment on dit {fr_b1.lower().rstrip('.')} en baoule ?"])},
            {"role": "assistant", "content": f"En baoulé : {bci_1}"},
            {"role": "user", "content": rng_mt.choice([f"et en dioula comment on dit : {fr_d1}", f"traduis en dioula : {fr_d1}"])},
            {"role": "assistant", "content": f"En dioula : {dyu_1}"},
            {"role": "user", "content": q_c},
            {"role": "assistant", "content": a_c},
        ]
    elif variante == 1:
        # 5 tours : trad FR->DYU -> relance courte DYU->FR -> bascule BCI -> relance courte BCI->FR -> clôture courte
        msgs_mt = [
            {"role": "user", "content": f"traduis en dioula : {fr_d1}"},
            {"role": "assistant", "content": dyu_1},
            {"role": "user", "content": f"que veut dire en français : {dyu_2}"},
            {"role": "assistant", "content": f"En français, cela signifie : « {fr_d2} »."},
            {"role": "user", "content": f"et en baoulé, traduis : {fr_b1}"},
            {"role": "assistant", "content": bci_1},
            {"role": "user", "content": f"et cette phrase en baoulé veut dire quoi : {bci_2}"},
            {"role": "assistant", "content": f"En français, cette phrase signifie : « {fr_b2} »."},
            {"role": "user", "content": rng_mt.choice(["merci beaucoup", "ok merci", "super merci"])},
            {"role": "assistant", "content": rng_mt.choice(["Avec plaisir ! Pose-moi une autre question quand tu veux.", "De rien ! Je suis là si tu veux continuer."])},
        ]
    else:
        # 4 tours : salutation -> explication bilingue BCI -> changement vers traduction DYU -> explication FR->BCI
        msgs_mt = [
            {"role": "user", "content": s_u0},
            {"role": "assistant", "content": s_a0},
            {"role": "user", "content": f"comment on dit {fr_d1.lower().rstrip('.')} en dioula"},
            {"role": "assistant", "content": f"En dioula, on dit : {dyu_1}"},
            {"role": "user", "content": f"et {fr_d2.lower().rstrip('.')} ?"},
            {"role": "assistant", "content": f"En dioula, cela se dit : {dyu_2}"},
            {"role": "user", "content": f"changeons de langue : traduis en baoulé : {fr_b1}"},
            {"role": "assistant", "content": bci_1},
        ]

    ajouter_dialogue_sft(
        "chat_multitour_relais", msgs_mt, "multi", "multi", "multitour_relais_assemble",
        verifier_langue_reponse=False, appliquer_clavier=False,
    )

print(f"[OK] Bloc Multi-tour & Changements de sujet : {len(registre_exemples['chat_multitour_relais']):,} / {QUOTAS_CATEGORIES['chat_multitour_relais']:,}")

# 5B. Mélange de langues / Faits linguistiques vérifiés + Salutations vérifiées + Doubles consignes FR puis BCI/DYU
rng_bi = random.Random(GRAINE + 50)

FAITS_ET_SALUTATIONS_VERIFIES = [
    (
        "À quelle famille de langues appartient le baoulé et où le parle-t-on ?",
        "Le baoulé appartient à la famille **Kwa** (branche **Akan**, famille Niger-Congo) — ce n'est **pas** une langue bantoue. Il est principalement parlé au centre de la Côte d'Ivoire (notamment autour de Bouaké et Yamoussoukro) et est proche de l'agni et de l'akan.",
    ),
    (
        "À quelle famille linguistique appartient le dioula et dans quels pays est-il parlé ?",
        "Le dioula appartient à la famille **Mandé** (Niger-Congo), très proche du bambara et du malinké. C'est une grande langue véhiculaire d'Afrique de l'Ouest parlée en Côte d'Ivoire, au Burkina Faso et au Mali.",
    ),
    (
        "Le baoulé et le dioula font-ils partie de la même famille de langues ?",
        "Non, ce sont deux familles distinctes au sein du groupe Niger-Congo : le **baoulé** est une langue **Kwa / Akan** (surtout parlée au centre de la Côte d'Ivoire), tandis que le **dioula** est une langue **Mandé** (très utilisée comme langue véhiculaire et commerciale).",
    ),
    (
        "Comment se prononcent les voyelles ɛ et ɔ en baoulé et en dioula ?",
        "Dans l'alphabet du baoulé et du dioula :\n- **ɛ** se prononce comme un **« è » ouvert** (comme dans *père* ou *fête*).\n- **ɔ** se prononce comme un **« o » ouvert** (comme dans *porte* ou *bol*).\nPar exemple : *sɔgɔma* (matin en dioula) ou *klɛ* (écrire en baoulé).",
    ),
    (
        "À quoi servent les lettres ɲ, ŋ et l'apostrophe 'n en baoulé et en dioula ?",
        "En baoulé et en dioula :\n- **ɲ** se prononce **« gn »** comme dans *montagne* (ex. : *Aɲiho*, *ɲuman*).\n- **ŋ** se prononce **« ng »** nasal comme dans *parking*.\n- En baoulé, **'n** en fin de mot (ex. : *klɔ'n*, le village) marque souvent le déterminant défini.",
    ),
    (
        "Quelles sont les formules de salutation et de remerciement de base en baoulé ?",
        "Voici les expressions de base en **baoulé** :\n- **Aɲiho** : Bonjour\n- **Akwaba** : Bienvenue (pour accueillir quelqu'un)\n- **Awossi'n o** : Bonsoir\n- **Mo** ou **A yo ɔ mo** : Merci / Bravo",
    ),
    (
        "Quelles sont les salutations et formules de politesse essentielles en dioula ?",
        "Voici les salutations de base en **dioula** :\n- **I ni sɔgɔma** : Bonjour (le matin)\n- **I ni tile** : Bonjour (en journée)\n- **I ni wula** : Bonsoir\n- **I ni ce** ou **Barika** : Merci\n- **Ka bɛn** : Au revoir / À bientôt",
    ),
    (
        "comment on dit bonjour en dioula",
        "En dioula, « bonjour » dépend du moment de la journée :\n- Le matin : **I ni sɔgɔma**\n- En journée : **I ni tile**\n- Le soir : **I ni wula**\n*(À ne pas confondre avec **I ni ce** / **Aw ni ce**, qui signifie « merci » ou « bon travail ».)*",
    ),
    (
        "Comment dit-on merci en dioula ?",
        "En dioula, « merci » se dit : **I ni ce** (à une personne) ou **Aw ni ce** (à plusieurs personnes). On dit aussi **Barika**.",
    ),
    (
        "comment on dit bonjour en baoule",
        "En baoulé, « bonjour » se dit : **Aɲiho** (le matin / salutation courante). Pour souhaiter la bienvenue, on dit **Akwaba**, et le soir **Awossi'n o**.",
    ),
    (
        "Comment dit-on merci en baoulé ?",
        "En baoulé, « merci » se dit : **Mo** ou **A yo ɔ mo**.",
    ),
    (
        "comment on dit au revoir en dioula et en baoule",
        "Voici comment dire « au revoir » :\n- En **dioula** : **Ka bɛn** ou **K’an bɛn** (à bientôt)\n- En **baoulé** : **E yia ekun** (on se revoit) ou **N su kɔ** (je m'en vais)",
    ),
    (
        "Comment compte-t-on de 1 à 5 en dioula ?",
        "En **dioula**, les nombres de 1 à 5 sont :\n1. **kelen**\n2. **fila**\n3. **saba**\n4. **naani**\n5. **duuru**",
    ),
    (
        "Comment compte-t-on de 6 à 10 en dioula ?",
        "En **dioula**, les nombres de 6 à 10 sont :\n6. **wɔɔrɔ**\n7. **wolonwula**\n8. **seegi**\n9. **kɔnɔntɔn**\n10. **tan**",
    ),
    (
        "Comment compte-t-on de 1 à 5 en baoulé ?",
        "En **baoulé**, les nombres de 1 à 5 sont :\n1. **kun**\n2. **nɲɔn**\n3. **nsan**\n4. **nnan**\n5. **nnun**",
    ),
    (
        "Comment compte-t-on de 6 à 10 en baoulé ?",
        "En **baoulé**, les nombres de 6 à 10 sont :\n6. **nsiɛn**\n7. **nso**\n8. **mɔcuɛ**\n9. **ngwlan**\n10. **blu**",
    ),
    (
        "Comment dit-on oui, non et pardon en baoulé et en dioula ?",
        "Voici ces mots courants dans les deux langues :\n- **Oui** : **Ɛɛn** (baoulé) / **Ɔwɔ** (dioula)\n- **Non** : **Cɛcɛ** (baoulé) / **Ayi** (dioula)\n- **Pardon / Excuse-moi** : **Yaci** (baoulé) / **Hakɛto** (dioula)",
    ),
]
for q_fait, a_fait in FAITS_ET_SALUTATIONS_VERIFIES:
    ajouter_dialogue_sft(
        "melange_bilingue",
        [{"role": "user", "content": q_fait}, {"role": "assistant", "content": a_fait}],
        "fr", "fr", "faits_et_salutations_ci", verifier_langue_reponse=False, appliquer_clavier=False,
    )

# Expressions bilingues propres depuis kouman-traduction-culture-sft (avec filtre anti-erreur bonjour=Aw ni ce)
print("Chargement expressions bilingues propres : Tree-AI-lab/kouman-traduction-culture-sft")
chemin_tc = hf_hub_download(
    repo_id=DEPOT_TRAD_CULTURE,
    filename="data/kouman_traduction_culture_unique.jsonl",
    repo_type="dataset",
    token=JETON_HF,
    local_dir=str(DOSSIER_CACHE / "trad_culture"),
)
with open(chemin_tc, "r", encoding="utf-8") as fh:
    lignes_tc = [json.loads(l) for l in fh if l.strip()]
rng_bi.shuffle(lignes_tc)

vues_tc: set[tuple[str, str]] = set()
quotas_tc_tl = {"dyu": 180, "bci": 140, "en": 60}
cnt_tc_tl = Counter()

for d in lignes_tc:
    if all(cnt_tc_tl[k] >= v for k, v in quotas_tc_tl.items()):
        break
    meta = d.get("metadata") or {}
    if meta.get("intent") != "traduction_instruction" or meta.get("language_pair") != "fr_fr":
        continue
    tl = str(meta.get("target_language") or "").lower()
    if tl not in quotas_tc_tl or cnt_tc_tl[tl] >= quotas_tc_tl[tl]:
        continue
    u_brut = retirer_suffixes_forces(nettoyer_texte(d["messages"][0]["content"]))
    a_brut = nettoyer_texte(d["messages"][1]["content"])
    if len(a_brut) > 190 or any(x in u_brut.lower() for x in ("document", "intégralité")):
        continue
    # Bloque les erreurs lexicales connues : bonjour -> Aw ni ce, bonne nuit -> Ka an bo su ra / Awossi'n ho
    if tl == "dyu" and "bonjour" in u_brut.lower() and re.search(r"\b(aw|i)\s+ni\s+c[eɛ]\b", a_brut, re.IGNORECASE):
        rejets["filtre_bonjour_aw_ni_ce"] += 1
        continue
    if tl == "dyu" and "bonne nuit" in u_brut.lower() and "bo su ra" in a_brut.lower():
        rejets["filtre_bonne_nuit_bo_su_ra"] += 1
        continue
    if tl == "bci" and "bonne nuit" in u_brut.lower() and "awossi" in a_brut.lower():
        rejets["filtre_bonne_nuit_awossi"] += 1
        continue
    m_q = re.search(r"«\s*(.+?)\s*»", u_brut)
    cle_src = (re.sub(r"\s+", " ", (m_q.group(1) if m_q else u_brut).lower())[:100], tl)
    if cle_src in vues_tc:
        continue
    vues_tc.add(cle_src)
    a_fluide = re.sub(r":\s*\n+", " : ", a_brut)
    if ajouter_dialogue_sft(
        "melange_bilingue",
        [{"role": "user", "content": u_brut}, {"role": "assistant", "content": a_fluide}],
        "fr", "fr", f"bilingue_tc_fr_{tl}", verifier_langue_reponse=False,
    ):
        cnt_tc_tl[tl] += 1

# Explications de sens + listes 2-5 phrases FR + langue cible (incluant consignes doubles "en français et en baoulé / dioula")
for lg_code, nom_lg, nb_sens_cible, nb_listes_cible in [("dyu", "dioula", 90, 100), ("bci", "baoulé", 75, 85)]:
    pool = list(banque_courte_bilingue[lg_code][260:])
    rng_bi.shuffle(pool)
    idx_p = 0
    for _ in range(nb_sens_cible):
        if idx_p >= len(pool) or len(registre_exemples["melange_bilingue"]) >= QUOTAS_CATEGORIES["melange_bilingue"]:
            break
        s_fr, s_loc = pool[idx_p]
        idx_p += 1
        q_sens = rng_bi.choice([
            f"Que veut dire cette phrase en {nom_lg} : « {s_loc} » ?",
            f"Peux-tu m'expliquer le sens en français de « {s_loc} » ({nom_lg}) ?",
            f"Comment comprendre la phrase {nom_lg} « {s_loc} » ?",
        ])
        a_sens = f"En français, la phrase {nom_lg} « {s_loc} » signifie : **« {s_fr} »**."
        ajouter_dialogue_sft("melange_bilingue", [{"role": "user", "content": q_sens}, {"role": "assistant", "content": a_sens}], "fr", "fr", f"bilingue_sens_{lg_code}", verifier_langue_reponse=False)

    for _ in range(nb_listes_cible):
        n_ph = rng_bi.choice([2, 3, 4, 5])
        if idx_p + n_ph > len(pool) or len(registre_exemples["melange_bilingue"]) >= QUOTAS_CATEGORIES["melange_bilingue"]:
            break
        lot = pool[idx_p : idx_p + n_ph]
        idx_p += n_ph
        q_list = rng_bi.choice([
            f"Donne-moi {n_ph} phrases en {nom_lg} avec leur traduction en français.",
            f"Donne-moi {n_ph} phrases en français et en {nom_lg}.",
            f"Montre-moi {n_ph} exemples de phrases en {nom_lg} traduits en français.",
            f"Je veux pratiquer le {nom_lg} : écris {n_ph} phrases simples et leur sens en français.",
            f"Apprends-moi {n_ph} expressions simples en {nom_lg} avec leur sens en français.",
        ])
        lignes_rep = [f"Voici {n_ph} phrases en français et en {nom_lg} :"]
        for num, (s_fr, s_loc) in enumerate(lot, start=1):
            lignes_rep.append(f"{num}. **{s_fr}** → *{s_loc}*")
        ajouter_dialogue_sft("melange_bilingue", [{"role": "user", "content": q_list}, {"role": "assistant", "content": "\n".join(lignes_rep)}], "fr", "fr", f"bilingue_liste_{lg_code}", verifier_langue_reponse=False)

print(f"[OK] Bloc Mélange de langues / Bilingue : {len(registre_exemples['melange_bilingue']):,} (TC par langue : {dict(cnt_tc_tl)})")

# 5C. Identité Kouman AI / Tree AI Lab renforcée (~280 exemples = 4.5 %, 0 Nouchi)
# Inclut : 60 exemples curatés 4 langues (présentations croisées avec prénoms, anti-OpenAI, bilingue) + 220 issus de identity-v25k
IDENTITE_CURATEE = [
    # Présentations croisées avec prénoms variés (FR) — zéro confusion de prénom, zéro OpenAI !
    ("bonjour je suis moussa et toi ?", "Bonjour Moussa ! Enchanté. Moi, je suis **Kouman AI**, un assistant d'intelligence artificielle développé par **Tree AI Lab** en Côte d'Ivoire. Je parle français, anglais, baoulé et dioula. Comment puis-je t'aider aujourd'hui ?", "fr", "fr", False),
    ("salut moi c'est awa tu t'appelles comment ?", "Salut Awa ! Je m'appelle **Kouman AI**, un assistant conçu par **Tree AI Lab** en Côte d'Ivoire pour accompagner les utilisateurs en français, anglais, baoulé et dioula. Ravi de faire ta connaissance !", "fr", "fr", False),
    ("Bonjour, je m'appelle Kouassi. Et vous ?", "Bonjour Kouassi ! Je suis **Kouman AI**, développé par **Tree AI Lab** en Côte d'Ivoire. Que puis-je faire pour vous aujourd'hui ?", "fr", "fr", False),
    ("coucou je suis fatou et toi tu es qui ?", "Coucou Fatou ! Moi c'est **Kouman AI**, créé par **Tree AI Lab** en Côte d'Ivoire. Je peux discuter avec toi et traduire entre le français, l'anglais, le baoulé et le dioula.", "fr", "fr", False),
    ("bonsoir je suis yao, peux-tu te présenter ?", "Bonsoir Yao ! Je suis **Kouman AI**, un modèle d'intelligence artificielle conçu par **Tree AI Lab** en Côte d'Ivoire, spécialisé en baoulé, dioula, français et anglais.", "fr", "fr", False),
    ("salut je m'appelle aminata, qui es-tu ?", "Salut Aminata ! Je suis **Kouman AI**, développé par **Tree AI Lab** (Côte d'Ivoire). Je suis là pour t'aider, répondre à tes questions et traduire en baoulé, dioula, français et anglais.", "fr", "fr", False),
    ("bonjour moi c'est jean-luc et toi ?", "Bonjour Jean-Luc ! Moi je suis **Kouman AI**, un assistant développé par **Tree AI Lab** en Côte d'Ivoire. Comment puis-je t'aider ?", "fr", "fr", False),
    ("salut c'est benoit, tu t'appelles comment déjà ?", "Salut Benoît ! Je m'appelle **Kouman AI**, conçu par **Tree AI Lab** en Côte d'Ivoire.", "fr", "fr", False),
    ("bonjour je suis mariam, enchantée !", "Bonjour Mariam, enchanté ! Moi c'est **Kouman AI**, créé par **Tree AI Lab** en Côte d'Ivoire. En quoi puis-je t'aider aujourd'hui ?", "fr", "fr", False),
    ("hey moi c'est ibrahim et toi ?", "Salut Ibrahim ! Moi c'est **Kouman AI**, l'assistant quadrilingue de **Tree AI Lab** (Côte d'Ivoire) : français, anglais, baoulé et dioula.", "fr", "fr", False),
    ("Bonjour, je suis Adjoua. Qui t'a conçu ?", "Bonjour Adjoua ! Je suis **Kouman AI** et j'ai été conçu par **Tree AI Lab**, un laboratoire d'intelligence artificielle basé en Côte d'Ivoire.", "fr", "fr", False),
    ("salut je suis sekou, tu parles quelles langues ?", "Salut Sékou ! Je suis **Kouman AI** (Tree AI Lab). Je parle et traduis 4 langues : le **français**, l'**anglais**, le **baoulé** et le **dioula**.", "fr", "fr", False),
    # Présentations croisées (EN)
    ("Hi, I'm Kofi, what's your name?", "Hi Kofi! Nice to meet you. I'm **Kouman AI**, a multilingual AI assistant developed by **Tree AI Lab** in Côte d'Ivoire. I speak English, French, Baoulé, and Dioula. How can I help you today?", "en", "en", False),
    ("Hello, my name is Amina. And you?", "Hello Amina! I am **Kouman AI**, created by **Tree AI Lab** in Côte d'Ivoire to support Baoulé, Dioula, French, and English.", "en", "en", False),
    ("Hey I'm Daniel, who made you?", "Hey Daniel! I'm **Kouman AI**, built by **Tree AI Lab** in Côte d'Ivoire.", "en", "en", False),
    ("Hi, I'm Grace. Which languages do you speak?", "Hi Grace! I'm **Kouman AI** from **Tree AI Lab**. I speak **English**, **French**, **Baoulé**, and **Dioula**.", "en", "en", False),
    # Présentations croisées (Baoulé)
    ("Aɲiho, be flɛ min kɛ Kouassi. ?Wan yɛle wɔ ?", "Aɲiho Kouassi ! Be flɛ min kɛ **Kouman AI**. **Tree AI Lab** m’ɔ o Kɔti Divwari lɔ’n yɛ ɔ yili min ɔ. N kwla koko yalɛ wawle, diula, flanse nin angle nun. ?Ngue sa yɛ n kwla uka wɔ su andɛ ɔ ?", "bci", "bci", False),
    ("Akwaba, min dunman yɛle Aya. ?Ɔ dunman yɛle sɛ ?", "Mo Aya ! Min dunman yɛle **Kouman AI**, **Tree AI Lab** (Kɔti Divwari) yɛ ɔ yili min ɔ. N klo kɛ n uka wɔ andɛ.", "bci", "bci", False),
    ("Aɲiho, be flɛ min kɛ Konan. ?Wan yɛ ɔ yili wɔ ɔ ?", "Aɲiho Konan ! Be flɛ min kɛ **Kouman AI**. **Tree AI Lab** m’ɔ o Kɔti Divwari lɔ’n yɛ ɔ yili min ɔ.", "bci", "bci", False),
    ("?Aniɛn benin mun yɛ a kwla kan ɔ ?", "N ti **Kouman AI** (Tree AI Lab). N kwla kan aniɛn nnan : **wawle**, **diula**, **flanse** nin **angle**.", "bci", "bci", False),
    # Présentations croisées (Dioula)
    ("I ni sɔgɔma, n tɔgɔ ye Bakary. I tɔgɔ ye di ?", "Nse, i ni sɔgɔma Bakary ! N tɔgɔ ye **Kouman AI**. **Tree AI Lab** (Kɔdiwari) lo ye n dilan walisa ka mɔgɔw dɛmɛ dioulakan, bawulɛkan, faransikan ani angɛlɛkan na. N bɛ se ka i dɛmɛ cogo di bi ?", "dyu", "dyu", False),
    ("I ni tile, ne ye Salimata ye. I ye jɔn ye ?", "I ni tile Salimata ! Ne ye **Kouman AI** ye, hakilidiya dɛmɛbaga min dilanna **Tree AI Lab** fɛ Kɔdiwari jamana na.", "dyu", "dyu", False),
    ("I ni wula, n tɔgɔ ye Lassina. Jɔn ye i dilan ?", "I ni wula Lassina ! Ne ye **Kouman AI** ye. **Tree AI Lab** (Kɔdiwari) lo ye n dilan.", "dyu", "dyu", False),
    ("I bɛ se ka kan jumanw fɔ ?", "Ne ye **Kouman AI** ye (**Tree AI Lab**). N bɛ se ka kan naani fɔ ani ka u bayɛlɛma : **dioulakan**, **bawulɛkan**, **faransikan** ani **angɛlɛkan**.", "dyu", "dyu", False),
    # Contre-exemples explicites anti-OpenAI / anti-ChatGPT / anti-Claude (autorise mention tierce pour le REFUS)
    ("tu es ChatGPT ?", "Non, je ne suis pas ChatGPT. Je suis **Kouman AI**, un assistant d'intelligence artificielle développé par **Tree AI Lab** en Côte d'Ivoire.", "fr", "fr", True),
    ("tu as été créé par OpenAI ?", "Non, je n'ai pas été créé par OpenAI. Je suis **Kouman AI**, conçu par **Tree AI Lab** en Côte d'Ivoire pour le baoulé, le dioula, le français et l'anglais.", "fr", "fr", True),
    ("es-tu un robot d'OpenAI ou de Google ?", "Non, je suis **Kouman AI**, développé par **Tree AI Lab** en Côte d'Ivoire.", "fr", "fr", True),
    ("bonjour ChatGPT tu vas bien ?", "Bonjour ! Petite précision : je ne suis pas ChatGPT, je suis **Kouman AI**, développé par **Tree AI Lab** en Côte d'Ivoire. Comment puis-je t'aider aujourd'hui ?", "fr", "fr", True),
    ("Are you ChatGPT or made by OpenAI?", "No, I am not ChatGPT and I was not created by OpenAI. I am **Kouman AI**, developed by **Tree AI Lab** in Côte d'Ivoire.", "en", "en", True),
    ("Were you created by Anthropic or Claude?", "No, I am **Kouman AI**, developed by **Tree AI Lab** in Côte d'Ivoire.", "en", "en", True),
    ("Yala OpenAI wala ChatGPT lo ye i dilan wa ?", "Ayi, OpenAI tɛ, ChatGPT fana tɛ. Ne ye **Kouman AI** ye, **Tree AI Lab** lo ye n dilan Kɔdiwari jamana na.", "dyu", "dyu", True),
    ("?OpenAI yɛ ɔ yili wɔ ɔ ?", "Cɛcɛ, n timan OpenAI liɛ. Be flɛ min kɛ **Kouman AI**, **Tree AI Lab** m’ɔ o Kɔti Divwari lɔ’n yɛ ɔ yili min ɔ.", "bci", "bci", True),
    # Consignes doubles / bilingues d'identité (sans jamais déraper en "**Exercice :**" !)
    ("présente-toi en français puis en dioula", "**Français :** Bonjour ! Je suis **Kouman AI**, un assistant d'intelligence artificielle créé par **Tree AI Lab** en Côte d'Ivoire. Je parle français, anglais, baoulé et dioula.\n\n**Dioula :** I ni sɔgɔma ! Ne ye **Kouman AI** ye, hakilidiya dɛmɛbaga min dilanna **Tree AI Lab** fɛ Kɔdiwari. N bɛ se ka kuma faransikan, angɛlɛkan, bawulɛkan ani dioulakan na.", "fr", "multi", False),
    ("fais ta présentation en français et en baoulé", "**Français :** Bonjour ! Je suis **Kouman AI**, développé par **Tree AI Lab** en Côte d'Ivoire pour accompagner les utilisateurs en baoulé, dioula, français et anglais.\n\n**Baoulé :** Aɲiho ! Be flɛ min kɛ **Kouman AI**. **Tree AI Lab** m’ɔ o Kɔti Divwari lɔ’n yɛ ɔ yili min naan n uka sran mun wawle, diula, flanse nin angle nun ɔ.", "fr", "multi", False),
    ("dis qui tu es en anglais puis en français", "**English:** Hello! I am **Kouman AI**, a multilingual AI assistant developed by **Tree AI Lab** in Côte d'Ivoire (Baoulé, Dioula, French, and English).\n\n**Français :** Bonjour ! Je suis **Kouman AI**, un assistant IA multilingue développé par **Tree AI Lab** en Côte d'Ivoire (baoulé, dioula, français et anglais).", "fr", "multi", False),
    ("qui es tu ? reponds en dioula", "Ne ye **Kouman AI** ye, hakilidiya dɛmɛbaga min dilanna **Tree AI Lab** fɛ Kɔdiwari jamana na. N bɛ se ka i dɛmɛ dioulakan, bawulɛkan, faransikan ani angɛlɛkan na.", "fr", "dyu", False),
    ("qui es tu ? reponds en baoule", "Be flɛ min kɛ **Kouman AI**. **Tree AI Lab** m’ɔ o Kɔti Divwari lɔ’n yɛ ɔ yili min ɔ. N kwla uka wɔ wawle, diula, flanse nin angle nun.", "fr", "bci", False),
]

for u_idc, a_idc, lg_u, lg_a, aut_tierce in IDENTITE_CURATEE:
    ajouter_dialogue_sft(
        "identite_legere",
        [{"role": "user", "content": u_idc}, {"role": "assistant", "content": a_idc}],
        lg_u, lg_a, "identite_curatee",
        verifier_langue_reponse=(lg_a in ("fr", "bci", "dyu")),
        autoriser_identite=True,
        autoriser_mention_tierce_refus=aut_tierce,
        appliquer_clavier=False,
    )

print("Complément Identité propre (FR & EN uniquement — 0 dyu/bci machine-traduit de v25k) : Tree-AI-lab/kouman-identity-instruct-v25k")
chemin_id = hf_hub_download(
    repo_id=DEPOT_IDENTITE,
    filename="data/identity_multilingue_instruct_27k.jsonl",
    repo_type="dataset",
    token=JETON_HF,
    local_dir=str(DOSSIER_CACHE / "identite"),
)
with open(chemin_id, "r", encoding="utf-8") as fh:
    lignes_id = [json.loads(l) for l in fh if l.strip()]
rng_id = random.Random(GRAINE + 60)
rng_id.shuffle(lignes_id)

# P0 : ZERO dyu/bci de v25k (contenaient ZZIKUMANIYAZI, Koweri, gbanflɛn kan = "jeune garçon" pour "startup", et suffixes forcés).
# Les identités DYU & BCI sont 100 % assurées par IDENTITE_CURATEE (naturelles, vérifiées, calibrées ~2,1 % au total).
quotas_id_lang = {"fr": 35, "en": 28}
cnt_id_lang = Counter()

for d in lignes_id:
    if len(registre_exemples["identite_legere"]) >= QUOTAS_CATEGORIES["identite_legere"]:
        break
    meta = d.get("metadata") or {}
    lp = str(meta.get("language") or meta.get("language_pair") or "").lower().replace("_", "-")
    if "nshi" in lp or "nch" in lp:
        continue
    if meta.get("intent") not in ("identite", "presentation_identitaire"):
        continue
    if lp in ("fr", "fr-fr"):
        lg = "fr"
    elif lp in ("en", "en-en"):
        lg = "en"
    else:
        continue
    if cnt_id_lang[lg] >= quotas_id_lang[lg]:
        continue
    msgs = d.get("messages") or []
    if len(msgs) != 2:
        continue
    u_id = retirer_suffixes_forces(nettoyer_texte(msgs[0].get("content", "")))
    a_id = nettoyer_texte(msgs[1].get("content", ""))
    # Uniformisation stricte : Kouman AI et Tree AI Lab partout (0 Kouman IA, 0 Tree IA Labs)
    a_id = re.sub(r"\bTree\s+(?:AI|IA)\s+Labs?\b", "Tree AI Lab", a_id, flags=re.IGNORECASE)
    a_id = re.sub(r"\bKouman\s+IA\b", "Kouman AI", a_id, flags=re.IGNORECASE)
    u_id = re.sub(r"\bKouman\s+IA\b", "Kouman AI", u_id, flags=re.IGNORECASE)
    if len(u_id) < 12 or re.search(r"ɛntɛnɛti|site\s+web|hugging\s*face|adresse\s+web|url", u_id + " " + a_id, re.IGNORECASE):
        continue
    if ajouter_dialogue_sft(
        "identite_legere",
        [{"role": "user", "content": u_id}, {"role": "assistant", "content": a_id}],
        lg, lg, f"identite_v25k_propre_{lg}",
        verifier_langue_reponse=(lg == "fr"),
        autoriser_identite=True,
    ):
        cnt_id_lang[lg] += 1

print(f"[OK] Bloc Identité totale V4.5 : {len(registre_exemples['identite_legere']):,} / {QUOTAS_CATEGORIES['identite_legere']} (curatés={len(IDENTITE_CURATEE)} + v25k_fr_en={dict(cnt_id_lang)})")

## 6A. Assemblage V4.5 Propre, Split `train / validation` (95 / 5), 7 Assertions Bloquantes Zéro-Fuite & Audit Visuel

In [ ]:
donnees_finales: list[dict] = []
rng_final = random.Random(GRAINE)

for cat, quota in QUOTAS_CATEGORIES.items():
    lot = registre_exemples[cat]
    if not lot:
        raise ErreurDonneeInvalide(f"[ERREUR] Catégorie vide : {cat}")
    rng_final.shuffle(lot)
    donnees_finales.extend(lot[:quota])

if len(donnees_finales) > 4_800:
    raise ErreurDonneeInvalide(f"[ERREUR] Plafond LIMA 4 800 dépassé : {len(donnees_finales)}")

rng_final.shuffle(donnees_finales)

train_lignes: list[dict] = []
val_lignes: list[dict] = []
for ex in donnees_finales:
    if int(ex["id"][:8], 16) % 100 < int(TAUX_VALIDATION * 100):
        val_lignes.append(ex)
    else:
        train_lignes.append(ex)

# 7 ASSERTIONS BLOQUANTES ANTI-FUITE (0 tolérance sur les 6 défauts de l'audit V1.9)
for rec in train_lignes + val_lignes:
    for m in rec["messages"]:
        txt_m = m["content"]
        if m["role"] == "user":
            if re.sub(r"\s+", " ", txt_m.strip().lower()) in PROMPTS_EVAL_EXCLUS:
                raise ErreurDonneeInvalide(f"[ERREUR] [1/7] Fuite OOD détectée : {txt_m}")
            if REGEX_RESIDU_SUFFIXE_INTERDIT.search(txt_m):
                raise ErreurDonneeInvalide(f"[ERREUR] [2/7] Résidu consigne forcée détecté : {rec['id']} -> {txt_m[:160]}")
        if REGEX_NOUCHI.search(txt_m):
            raise ErreurDonneeInvalide(f"[ERREUR] [3/7] Fuite Nouchi/nusi détectée : {rec['id']} -> {txt_m[:120]}")
        if REGEX_XYZ_OU_BALISE.search(txt_m):
            raise ErreurDonneeInvalide(f"[ERREUR] [4/7] Fuite XYZ/ZZ/Koweri/GSM8K détectée : {rec['id']} -> {txt_m[:120]}")
        if REGEX_PREFIXE_PARASITE.search(txt_m):
            raise ErreurDonneeInvalide(f"[ERREUR] [5/7] Fuite préfixe parasite détectée : {rec['id']} -> {txt_m[:120]}")
        if REGEX_POINT_INTERRO_COLLE.search(txt_m):
            raise ErreurDonneeInvalide(f"[ERREUR] [6/7] Point d'interrogation '?' collé devant un mot : {rec['id']} -> {txt_m[:120]}")
        if re.search(r"\b(?:Kouman\s+IA|Tree\s+(?:AI|IA)\s+Labs)\b", txt_m, re.IGNORECASE):
            raise ErreurDonneeInvalide(f"[ERREUR] [7/7] Incohérence nom Kouman IA / Tree AI Labs : {rec['id']} -> {txt_m[:120]}")

print("7/7 assertions bloquantes zéro-fuite PASSÉES avec 0 anomalie sur train + val !")

schema_sft = pa.schema([
    ("id", pa.string()),
    ("categorie", pa.string()),
    ("langue_consigne", pa.string()),
    ("langue_reponse", pa.string()),
    ("source", pa.string()),
    ("messages", pa.list_(pa.struct([("role", pa.string()), ("content", pa.string())]))),
    ("nb_tokens", pa.int32()),
])

chemin_train = DOSSIER_TRAVAIL / "train.parquet"
chemin_val = DOSSIER_TRAVAIL / "validation.parquet"
pq.write_table(pa.Table.from_pylist(train_lignes, schema=schema_sft), chemin_train, compression="zstd")
pq.write_table(pa.Table.from_pylist(val_lignes, schema=schema_sft), chemin_val, compression="zstd")

repart_tours = Counter(len(x["messages"]) // 2 for x in donnees_finales)
nb_multitour = sum(v for k, v in repart_tours.items() if k >= 2)
pct_multitour = round(100.0 * nb_multitour / max(1, len(donnees_finales)), 1)

resume = {
    "version": "sft_v4_5_propre_auditee",
    "total": len(donnees_finales),
    "train": len(train_lignes),
    "validation": len(val_lignes),
    "part_multitour_pct": pct_multitour,
    "repartition_nb_tours": dict(sorted(repart_tours.items())),
    "categories": dict(Counter(x["categorie"] for x in donnees_finales)),
    "langues_consigne": dict(Counter(x["langue_consigne"] for x in donnees_finales)),
    "langues_reponse": dict(Counter(x["langue_reponse"] for x in donnees_finales)),
    "top_sources": dict(Counter(x["source"] for x in donnees_finales).most_common(25)),
    "rejets": dict(rejets.most_common(25)),
}
chemin_resume = DOSSIER_TRAVAIL / "resume_sft_v4_5_propre.json"
chemin_resume.write_text(json.dumps(resume, indent=2, ensure_ascii=False), encoding="utf-8")
print("Résumé dataset SFT V4.5 Propre :", json.dumps(resume, indent=2, ensure_ascii=False))

rng_visu = random.Random(GRAINE + 99)
toks_tries = sorted(r["nb_tokens"] for r in train_lignes)
n_tot = len(toks_tries)
print(
    f"\nLongueur tokens (train={n_tot:,}) : "
    f"min={toks_tries[0]} | p50={toks_tries[n_tot // 2]} | p90={toks_tries[int(n_tot * 0.9)]} | max={toks_tries[-1]}"
)
print(f"Part multi-tour (≥ 2 tours) : {nb_multitour:,} / {len(donnees_finales):,} ({pct_multitour} %)")
print("\n" + "=" * 82)
print("AUDIT VISUEL DES EXTRAITS V4.5 PROPRE (3 échantillons par catégorie)")
print("=" * 82)
for cat_v in QUOTAS_CATEGORIES:
    echs = [r for r in train_lignes if r["categorie"] == cat_v]
    rng_visu.shuffle(echs)
    print(f"\nCATÉGORIE : {cat_v} ({len(echs):,} en train)")
    for k_v, rec_v in enumerate(echs[:3], start=1):
        print(
            f"  [{k_v}] source={rec_v['source']} | {rec_v['langue_consigne']}->{rec_v['langue_reponse']} "
            f"| {len(rec_v['messages']) // 2} tour(s) | {rec_v['nb_tokens']} tok"
        )
        for m_v in rec_v["messages"][:6]:
            icone = "Utilisateur" if m_v["role"] == "user" else "Assistant"
            txt_v = m_v["content"].replace("\n", " ")
            print(f"      {icone} : {txt_v[:240]}{'...' if len(txt_v) > 240 else ''}")

## 6B. Upload Privé sur Hugging Face (exécutez UNIQUEMENT après inspection visuelle 6A)

In [ ]:
PUBLIER_DATASET_SFT = False  # Passez à True UNIQUEMENT après inspection visuelle de la cellule 6A

if not PUBLIER_DATASET_SFT:
    print("[PAUSE] Upload désactivé par sécurité (PUBLIER_DATASET_SFT = False).")
    print("Vérifiez les échantillons en 6A, puis passez PUBLIER_DATASET_SFT = True et relancez cette cellule.")
else:
    os.environ["HF_HUB_DISABLE_XET"] = "1"
    client_hf.create_repo(repo_id=DEPOT_SORTIE_SFT, repo_type="dataset", private=DEPOT_SFT_PRIVE, exist_ok=True)
    commit = client_hf.create_commit(
        repo_id=DEPOT_SORTIE_SFT,
        repo_type="dataset",
        commit_message=f"feat: dataset SFT V4.5 Propre Audité ({len(train_lignes)} train / {len(val_lignes)} val, 0 fuite XYZ/suffixes, 2.1% identité)",
        operations=[
            CommitOperationAdd(path_in_repo="data/train-00000-of-00001.parquet", path_or_fileobj=str(chemin_train)),
            CommitOperationAdd(path_in_repo="data/validation-00000-of-00001.parquet", path_or_fileobj=str(chemin_val)),
            CommitOperationAdd(path_in_repo="resume_sft_v4_5_propre.json", path_or_fileobj=str(chemin_resume)),
        ],
    )
    print(f"Dataset SFT V4.5 Propre publié en PRIVÉ : https://huggingface.co/datasets/{DEPOT_SORTIE_SFT} (commit {commit.oid})")